In [1]:
!pip install fairlearn pgmpy optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 33.4 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import os
import numpy as np
import pandas as pd
import joblib
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.compose import ColumnTransformer

CACHE_DIR = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42


def clean_numeric_column(series):
    med = pd.to_numeric(series, errors="coerce")
    return med.fillna(med.median() if med.notna().any() else 0)


def safe_qcut(series, q):
    try:
        return pd.qcut(series, q=q, labels=False, duplicates="drop").fillna(0).astype(int)
    except Exception:
        return pd.cut(series, bins=q, labels=False, duplicates="drop").fillna(0).astype(int)


def four_way_balance(X, y, sensitive, rng=None):
    if rng is None:
        rng = np.random.RandomState(SEED)
    groups = np.unique(sensitive)
    if len(groups) != 2:
        return X, y, sensitive
    counts = {}
    for g in groups:
        for lbl in [0, 1]:
            mask = (sensitive == g) & (y == lbl)
            counts[(g, lbl)] = mask.sum()
    target_count = max(counts.values())
    all_idx = []
    base_idx = np.arange(len(y))
    for g in groups:
        for lbl in [0, 1]:
            mask = (sensitive == g) & (y == lbl)
            idx = base_idx[mask]
            deficit = target_count - len(idx)
            if deficit > 0 and len(idx) > 0:
                extra = rng.choice(idx, size=deficit, replace=True)
                idx = np.concatenate([idx, extra])
            all_idx.append(idx)
    full_idx = np.concatenate(all_idx)
    rng.shuffle(full_idx)
    return X[full_idx], y[full_idx], sensitive[full_idx]


def stratified_group_split(X, y, sens_list, test_size=0.2, seed=SEED):
    strat_key = y.astype(str)
    for s in sens_list:
        strat_key = strat_key + "_" + np.array(s).astype(str)
    unique_keys, counts = np.unique(strat_key, return_counts=True)
    small_groups = unique_keys[counts < 2]
    mask_small = np.isin(strat_key, small_groups)
    strat_key[mask_small] = y[mask_small].astype(str)
    try:
        return train_test_split(
            np.arange(len(y)), test_size=test_size, stratify=strat_key, random_state=seed
        )
    except Exception:
        return train_test_split(
            np.arange(len(y)), test_size=test_size, stratify=y, random_state=seed
        )


def preprocess_compas_for_fair_bbn(
    csv_path="/kaggle/input/datasets/maansitemp/all-datasets/compas-scores-two-years.csv",
    seed=42,
    use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, "preproc_compas_v4.pkl")
    if use_cache and os.path.exists(cache_file):
        print("Loading cached COMPAS data...")
        return joblib.load(cache_file)

    df = pd.read_csv(csv_path)
    df = df.loc[
        (df["days_b_screening_arrest"] <= 30)
        & (df["days_b_screening_arrest"] >= -30)
        & (df["is_recid"] != -1)
        & (df["c_charge_degree"] != "O")
        & (df["score_text"] != "N/A"),
        [
            "age", "c_charge_degree", "race", "age_cat",
            "sex", "priors_count", "days_b_screening_arrest",
            "juv_other_count", "juv_misd_count", "juv_fel_count",
            "c_charge_desc", "is_recid", "two_year_recid",
            "c_jail_in", "c_jail_out",
        ],
    ].reset_index(drop=True)

    # Binary race: African-American=1 vs rest=0 (standard COMPAS fairness split)
    df["race_binary"] = (df["race"] == "African-American").astype(int)
    # Binary sex: Male=0, Female=1
    df["sex_binary"] = (df["sex"] == "Female").astype(int)

    df["c_jail_in"] = pd.to_datetime(df["c_jail_in"], errors="coerce")
    df["c_jail_out"] = pd.to_datetime(df["c_jail_out"], errors="coerce")
    df["jail_time"] = (df["c_jail_out"] - df["c_jail_in"]).dt.days.fillna(0).clip(lower=0)
    df = df.drop(columns=["c_jail_in", "c_jail_out"])

    # Engineered features (NO decile_score or score_text — those embed the biased COMPAS score)
    df["priors_count_sq"] = df["priors_count"] ** 2
    df["age_x_priors"] = df["age"] * df["priors_count"]
    df["juv_total"] = df["juv_other_count"] + df["juv_misd_count"] + df["juv_fel_count"]
    df["any_juv"] = (df["juv_total"] > 0).astype(int)
    df["young_adult"] = (df["age"] < 25).astype(int)
    df["long_jail"] = (df["jail_time"] > 30).astype(int)
    df["log_priors"] = np.log1p(df["priors_count"])

    y = df["two_year_recid"].values
    sens_race = df["race_binary"].values   # binary: 1=African-American
    sens_sex = df["sex_binary"].values     # binary: 1=Female

    numeric_cols = [
        "age", "priors_count", "days_b_screening_arrest", "jail_time",
        "juv_other_count", "juv_misd_count", "juv_fel_count",
        "priors_count_sq", "age_x_priors", "juv_total",
        "any_juv", "young_adult", "long_jail", "log_priors",
    ]
    categorical_cols = ["c_charge_degree", "race", "age_cat", "sex", "c_charge_desc"]

    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])

    preproc = ColumnTransformer(
        [
            ("num", Pipeline([("scaler", StandardScaler())]), numeric_cols),
            ("cat", Pipeline([("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_cols),
        ]
    )

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ["race", "sex"]:
        bbn_df[col] = bbn_df[col].astype("category").cat.codes.astype(int)

    X = df.drop(columns=["is_recid", "two_year_recid", "race_binary", "sex_binary"])

    train_idx, test_idx = stratified_group_split(
        X.values, y, [sens_race, sens_sex], test_size=0.2, seed=seed
    )

    X_train_raw = X.iloc[train_idx].reset_index(drop=True)
    X_test_raw = X.iloc[test_idx].reset_index(drop=True)
    y_train, y_test = y[train_idx], y[test_idx]
    sens_race_train, sens_race_test = sens_race[train_idx], sens_race[test_idx]
    sens_sex_train, sens_sex_test = sens_sex[train_idx], sens_sex[test_idx]

    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)

    bbn_train = bbn_df.iloc[train_idx].reset_index(drop=True)
    bbn_test = bbn_df.iloc[test_idx].reset_index(drop=True)

    result = {
        "X_train_nn": X_train_nn, "X_test_nn": X_test_nn,
        "bbn_train_df": bbn_train, "bbn_test_df": bbn_test,
        "y_train": y_train, "y_test": y_test,
        "sens_race_train": pd.Series(sens_race_train),
        "sens_race_test": pd.Series(sens_race_test),
        "sens_sex_train": pd.Series(sens_sex_train),
        "sens_sex_test": pd.Series(sens_sex_test),
    }

    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached COMPAS data to {cache_file}")

    return result


def preprocess_german_for_fair_bbn(
    csv_path="/kaggle/input/datasets/maansitemp/all-datasets/german.data",
    seed=42,
    use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, "preproc_german_v4.pkl")
    if use_cache and os.path.exists(cache_file):
        print("Loading cached German data...")
        return joblib.load(cache_file)

    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings",
        "employment", "installment_rate", "personal_status_sex", "other_debtors",
        "residence", "property", "age", "other_installment_plans", "housing",
        "number_credits", "job", "people_liable", "telephone", "foreign_worker", "credit",
    ]
    df = pd.read_csv(csv_path, sep=" ", names=column_names)

    sex_map = {"A91": "male", "A92": "male", "A93": "male", "A94": "female", "A95": "female"}
    df["sex"] = df["personal_status_sex"].map(sex_map)
    df["sex_binary"] = df["sex"].map({"male": 0, "female": 1}).fillna(0).astype(int)
    df["age_binary"] = (df["age"] >= 25).astype(int)   # 1=older, 0=young
    df["credit_label"] = df["credit"].map({1: 1, 2: 0})
    df = df.drop(columns=["personal_status_sex", "sex", "foreign_worker", "credit"])

    df["log_amount"] = np.log1p(df["amount"].clip(lower=0))
    df["amount_x_duration"] = df["amount"] * df["duration"]
    df["high_installment"] = (df["installment_rate"] >= 3).astype(int)
    df["long_duration"] = (df["duration"] > 24).astype(int)
    df["amount_per_month"] = df["amount"] / df["duration"].clip(lower=1)
    df["young"] = (df["age"] < 25).astype(int)

    y = df["credit_label"].values
    sensitive_sex = df["sex_binary"].values
    sensitive_age = df["age_binary"].values

    numerical_cols = [
        "duration", "amount", "installment_rate", "residence",
        "number_credits", "people_liable", "age",
        "log_amount", "amount_x_duration", "high_installment",
        "long_duration", "amount_per_month", "young",
    ]
    categorical_cols = [c for c in df.columns if c not in numerical_cols + [
        "credit_label", "sex_binary", "age_binary"
    ]]

    for col in numerical_cols:
        df[col] = clean_numeric_column(df[col])

    preproc = ColumnTransformer(
        [
            ("num", StandardScaler(), numerical_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ]
    )

    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ["sex_binary", "age_binary"]:
        bbn_df[col] = bbn_df[col].astype("category").cat.codes.astype(int)

    X = df.drop(columns=["credit_label"])

    train_idx, test_idx = stratified_group_split(
        X.values, y, [sensitive_age, sensitive_sex], test_size=0.2, seed=seed
    )

    X_train_raw = X.iloc[train_idx].reset_index(drop=True)
    X_test_raw = X.iloc[test_idx].reset_index(drop=True)
    y_train, y_test = y[train_idx], y[test_idx]
    sens_age_train, sens_age_test = sensitive_age[train_idx], sensitive_age[test_idx]
    sens_sex_train, sens_sex_test = sensitive_sex[train_idx], sensitive_sex[test_idx]

    pipeline = Pipeline([("preprocessor", preproc)])
    X_train_nn = pipeline.fit_transform(X_train_raw)
    X_test_nn = pipeline.transform(X_test_raw)

    bbn_train_df = bbn_df.iloc[train_idx].reset_index(drop=True)
    bbn_test_df = bbn_df.iloc[test_idx].reset_index(drop=True)

    result = {
        "X_train_nn": X_train_nn, "X_test_nn": X_test_nn,
        "bbn_train_df": bbn_train_df, "bbn_test_df": bbn_test_df,
        "y_train": y_train, "y_test": y_test,
        "sens_age_train": sens_age_train, "sens_age_test": sens_age_test,
        "sens_sex_train": sens_sex_train, "sens_sex_test": sens_sex_test,
    }

    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached German data to {cache_file}")

    return result


def preprocess_lawschool_for_fair_bbn(
    law_path="/kaggle/input/datasets/maansitemp/all-datasets/bar_pass_prediction.csv",
    seed=42,
    use_cache=True,
):
    """
    Target: pass_bar (bar passage) — the fairness-relevant outcome.

    BUG FIXES vs v4:
    1. Race binarization MUST happen BEFORE clean_numeric_column().
       clean_numeric_column() casts the race column to float ("white" → NaN → 0.0),
       so the subsequent string-match for "white" always fails → race_binary=0 everywhere.
       Fix: extract race_raw (string) immediately after loading, before any numeric cleaning.
    2. The bar_pass_prediction race column is often numeric (1=White, 2=Black, 3=Hispanic…).
       Handle both the numeric-code AND string variants so neither silently zeroes out.
    3. Gender: the 'sex' column is sometimes stored as 1/2 (male/female) integers — keep the
       original string/code before numeric imputation to get a valid binary.
    """
    cache_file = os.path.join(CACHE_DIR, "preproc_lawschool_v5.pkl")
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Law School data...")
        return joblib.load(cache_file)

    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]

    target_col = "pass_bar"
    sens_race_col = "race"
    sens_gender_col = "sex"

    law_df = law_df.dropna(subset=[target_col, sens_race_col, sens_gender_col]).reset_index(drop=True)

    # ── Step 1: extract sensitive attributes BEFORE any numeric coercion ─────
    # Race: save raw values as string right after loading
    race_raw = law_df[sens_race_col].astype(str).str.strip()

    # Determine encoding: numeric codes (e.g. "1", "1.0") vs string labels ("white")
    # In bar_pass_prediction the typical numeric encoding is:
    #   1 = White, 2 = Black, 3 = Hispanic, 4 = Asian, 5 = Native American, 6 = Puerto Rican, 7 = Other
    def _binarize_race(val):
        v = val.lower().strip()
        # String label variant
        if "white" in v:
            return 1
        # Numeric code variant — 1 or 1.0 means White
        try:
            code = int(float(v))
            return 1 if code == 1 else 0
        except (ValueError, OverflowError):
            return 0

    law_df["race_binary"] = race_raw.apply(_binarize_race)

    # Gender: save raw before numeric cleaning
    gender_raw = law_df[sens_gender_col].astype(str).str.strip().str.lower()

    def _binarize_gender(val):
        # String variants: "female", "f", "2" (if 1=male, 2=female)
        if val in ("female", "f", "2", "2.0"):
            return 1
        if val in ("male", "m", "1", "1.0"):
            return 0
        # Fallback: category code
        try:
            return int(float(val)) % 2  # handles arbitrary numeric codes
        except (ValueError, OverflowError):
            return 0

    law_df["gender_binary"] = gender_raw.apply(_binarize_gender)
    # Ensure strictly binary
    if law_df["gender_binary"].nunique() > 2:
        law_df["gender_binary"] = (law_df["gender_binary"] > 0).astype(int)

    # ── Step 2: binarize target ───────────────────────────────────────────────
    law_df[target_col] = (pd.to_numeric(law_df[target_col], errors="coerce") >= 1).astype(int)
    law_df = law_df.dropna(subset=[target_col]).reset_index(drop=True)

    # ── Step 3: numeric cleaning (AFTER sensitive attrs are extracted) ────────
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = clean_numeric_column(law_df[col])

    numeric_cols = [c for c in ["lsat", "ugpa", "zfygpa", "zgpa", "gpa", "fam_inc"] if c in law_df.columns]
    categorical_cols = [c for c in ["tier", "cluster", "fulltime", "parttime"] if c in law_df.columns]

    low_var_cols = [col for col in numeric_cols if law_df[col].nunique() <= 1]
    if low_var_cols:
        law_df = law_df.drop(columns=low_var_cols)
        numeric_cols = [c for c in numeric_cols if c not in low_var_cols]

    if "lsat" in law_df.columns and "ugpa" in law_df.columns:
        law_df["lsat_x_gpa"] = law_df["lsat"] * law_df["ugpa"]
        law_df["lsat_sq"] = law_df["lsat"] ** 2
        numeric_cols = numeric_cols + ["lsat_x_gpa", "lsat_sq"]

    preproc = ColumnTransformer(
        [
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ]
    )

    bbn_df = law_df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(law_df[col], 5)
    for col in categorical_cols + [sens_race_col, sens_gender_col]:
        bbn_df[col] = bbn_df[col].astype("category").cat.codes.astype(int)

    sens_race_labels = law_df["race_binary"].values
    sens_gender_labels = law_df["gender_binary"].values

    X = law_df[numeric_cols + categorical_cols + [sens_race_col, sens_gender_col]]
    y = law_df[target_col].values.astype(int)

    # ── Sanity checks — catch silent encoding failures early ─────────────────
    pos_rate = y.mean()
    race_mean = sens_race_labels.mean()
    gender_mean = sens_gender_labels.mean()
    print(f"  [Lawschool] pass_bar rate={pos_rate:.3f}, n={len(y)}, "
          f"race_binary mean={race_mean:.3f} (white frac), "
          f"gender_binary mean={gender_mean:.3f} (female frac)")
    print(f"  [Lawschool] race value counts: {dict(zip(*np.unique(sens_race_labels, return_counts=True)))}")
    print(f"  [Lawschool] gender value counts: {dict(zip(*np.unique(sens_gender_labels, return_counts=True)))}")

    if race_mean == 0.0:
        raise ValueError(
            "[Lawschool] race_binary is all zeros — race encoding failed. "
            f"Sample race column values: {law_df[sens_race_col].head(10).tolist()}"
        )
    if gender_mean == 0.0 or gender_mean == 1.0:
        raise ValueError(
            "[Lawschool] gender_binary is constant — gender encoding failed. "
            f"Sample sex column values: {law_df[sens_gender_col].head(10).tolist()}"
        )

    train_idx, test_idx = stratified_group_split(
        X.values, y, [sens_race_labels, sens_gender_labels], test_size=0.2, seed=seed
    )

    X_train_raw = X.iloc[train_idx].reset_index(drop=True)
    X_test_raw = X.iloc[test_idx].reset_index(drop=True)
    y_train, y_test = y[train_idx], y[test_idx]
    sens_race_train, sens_race_test = sens_race_labels[train_idx], sens_race_labels[test_idx]
    sens_gender_train, sens_gender_test = sens_gender_labels[train_idx], sens_gender_labels[test_idx]

    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)

    bbn_train = bbn_df.iloc[train_idx].reset_index(drop=True)
    bbn_test = bbn_df.iloc[test_idx].reset_index(drop=True)

    result = {
        "X_train_nn": X_train_nn, "X_test_nn": X_test_nn,
        "bbn_train_df": bbn_train, "bbn_test_df": bbn_test,
        "y_train": y_train, "y_test": y_test,
        "sens_race_train": pd.Series(sens_race_train),
        "sens_race_test": pd.Series(sens_race_test),
        "sens_gender_train": pd.Series(sens_gender_train),
        "sens_gender_test": pd.Series(sens_gender_test),
    }

    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Law School data to {cache_file}")

    return result


def preprocess_diabetes_hospital_for_fair_bbn(
    csv_path="/kaggle/input/datasets/maansitemp/all-datasets/diabetes_hospital_fairlearn.csv",
    seed=42,
    use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, "preproc_hospital_v4.pkl")
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Hospital data...")
        return joblib.load(cache_file)

    df = pd.read_csv(csv_path)
    drop_cols = ["max_glu_serum", "A1Cresult", "readmitted.1"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[~df.isin(["Missing"]).any(axis=1)]
    df = df.dropna(subset=["race", "gender"]).reset_index(drop=True)

    age_map = {
        "'0-10'": 5, "'10-20'": 15, "'20-30'": 25, "'30-40'": 35, "'40-50'": 45,
        "'50-60'": 55, "'60-70'": 65, "'70-80'": 75, "'80-90'": 85, "'90-100'": 95,
        "'30 years or younger'": 20, "'30-60 years'": 45, "'Over 60 years'": 70,
    }
    df["age"] = df["age"].replace(age_map).astype(float)
    df["readmit_binary"] = df["readmit_binary"].astype(int)
    df["race"] = df["race"].astype(str).str.strip()
    df["gender"] = df["gender"].astype(str).str.strip()

    # Binary race: Caucasian=0 vs non-Caucasian=1
    df["race_binary"] = (df["race"] != "Caucasian").astype(int)
    # Binary gender: Male=0, Female=1
    df["gender_binary"] = df["gender"].map({"Male": 0, "Female": 1}).fillna(0).astype(int)

    categorical_cols = [
        c for c in [
            "discharge_disposition_id", "admission_source_id",
            "medical_specialty", "primary_diagnosis",
            "insulin", "change", "diabetesMed",
        ]
        if c in df.columns
    ]
    numeric_cols = [
        c for c in [
            "age", "time_in_hospital", "num_lab_procedures", "num_procedures",
            "num_medications", "number_diagnoses", "had_emergency",
            "had_inpatient_days", "had_outpatient_days", "medicare", "medicaid",
        ]
        if c in df.columns
    ]

    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])

    if "num_medications" in df.columns and "num_lab_procedures" in df.columns:
        df["meds_x_labs"] = df["num_medications"] * df["num_lab_procedures"]
        df["total_procedures"] = df["num_lab_procedures"] + df.get("num_procedures", pd.Series(0, index=df.index))
        numeric_cols = numeric_cols + ["meds_x_labs", "total_procedures"]

    if "time_in_hospital" in df.columns:
        df["long_stay"] = (df["time_in_hospital"] > 7).astype(int)
        numeric_cols.append("long_stay")

    for col in ["num_medications", "num_lab_procedures", "time_in_hospital"]:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))

    preproc = ColumnTransformer(
        [
            ("num", Pipeline([("scaler", StandardScaler())]), numeric_cols),
            ("cat", Pipeline([("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_cols),
        ]
    )

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ["race", "gender"]:
        bbn_df[col] = bbn_df[col].astype("category").cat.codes.astype(int)

    X = df.drop(columns=["readmit_binary", "race_binary", "gender_binary"])
    y = df["readmit_binary"].values
    sens_race = df["race_binary"].values
    sens_gender = df["gender_binary"].values

    train_idx, test_idx = stratified_group_split(
        X.values, y, [sens_race, sens_gender], test_size=0.2, seed=seed
    )

    X_train_raw = X.iloc[train_idx].reset_index(drop=True)
    X_test_raw = X.iloc[test_idx].reset_index(drop=True)
    y_train, y_test = y[train_idx], y[test_idx]
    sens_race_train, sens_race_test = sens_race[train_idx], sens_race[test_idx]
    sens_gender_train, sens_gender_test = sens_gender[train_idx], sens_gender[test_idx]

    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)

    bbn_train = bbn_df.iloc[train_idx].reset_index(drop=True)
    bbn_test = bbn_df.iloc[test_idx].reset_index(drop=True)

    result = {
        "X_train_nn": X_train_nn, "X_test_nn": X_test_nn,
        "bbn_train_df": bbn_train, "bbn_test_df": bbn_test,
        "y_train": y_train, "y_test": y_test,
        "sens_race_train": pd.Series(sens_race_train),
        "sens_race_test": pd.Series(sens_race_test),
        "sens_sex_train": pd.Series(sens_gender_train),
        "sens_sex_test": pd.Series(sens_gender_test),
    }

    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Hospital data to {cache_file}")

    return result


def preprocess_bank_for_fair_bbn(
    csv_path="/kaggle/input/datasets/maansitemp/all-datasets/bank-full.csv",
    seed=42,
    use_cache=True,
):
    cache_file = os.path.join(CACHE_DIR, "preproc_bank_v4.pkl")
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Bank data...")
        return joblib.load(cache_file)

    try:
        df = pd.read_csv(csv_path, sep=";")
    except Exception:
        df = pd.read_csv(csv_path, sep=",")

    df = df.apply(lambda x: x.str.lower() if x.dtype == "object" else x)
    target_col = "y" if "y" in df.columns else ("deposit" if "deposit" in df.columns else "subscribed")
    if target_col not in df.columns:
        target_col = df.columns[-1]

    df = df[~df.isin(["unknown"]).any(axis=1)].reset_index(drop=True)
    df["y"] = np.where(df[target_col].isin(["yes", "y", "true", "1"]), 1, 0)

    le_marital = LabelEncoder()
    df["marital_label"] = le_marital.fit_transform(df["marital"])
    # Binarize marital: married=1 vs others=0
    df["marital_binary"] = (df["marital"] == "married").astype(int)

    le_job = LabelEncoder()
    df["job_label"] = le_job.fit_transform(df["job"])
    # Binarize job: blue-collar/services vs white-collar/professional
    blue_collar = ["blue-collar", "services", "technician", "admin.", "housemaid"]
    df["job_binary"] = df["job"].apply(lambda x: 1 if x in blue_collar else 0)

    categorical_cols = [
        c for c in ["job", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
        if c in df.columns
    ]
    numeric_cols = [
        c for c in ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
        if c in df.columns
    ]

    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])

    for col in ["balance", "duration", "pdays", "previous"]:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))

    if "balance" in df.columns:
        df["log_balance"] = np.log1p(df["balance"].clip(lower=0))
        df["neg_balance"] = (df["balance"] < 0).astype(int)
        numeric_cols = numeric_cols + ["log_balance", "neg_balance"]

    if "campaign" in df.columns:
        df["high_campaign"] = (df["campaign"] > 3).astype(int)
        numeric_cols.append("high_campaign")

    if "duration" in df.columns:
        df["log_duration"] = np.log1p(df["duration"])
        numeric_cols.append("log_duration")

    preproc = ColumnTransformer(
        [
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ]
    )

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ["marital", "job"]:
        bbn_df[col] = bbn_df[col].astype("category").cat.codes.astype(int)

    X = df.drop(columns=["y", "marital_label", "job_label", "marital_binary", "job_binary"])
    y = df["y"].values
    sens_marital = df["marital_binary"].values
    sens_job = df["job_binary"].values

    train_idx, test_idx = stratified_group_split(
        X.values, y, [sens_marital, sens_job], test_size=0.2, seed=seed
    )

    X_train_raw = X.iloc[train_idx].reset_index(drop=True)
    X_test_raw = X.iloc[test_idx].reset_index(drop=True)
    y_train, y_test = y[train_idx], y[test_idx]
    sens_marital_train, sens_marital_test = sens_marital[train_idx], sens_marital[test_idx]
    sens_job_train, sens_job_test = sens_job[train_idx], sens_job[test_idx]

    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)

    bbn_train = bbn_df.iloc[train_idx].reset_index(drop=True)
    bbn_test = bbn_df.iloc[test_idx].reset_index(drop=True)

    result = {
        "X_train_nn": X_train_nn, "X_test_nn": X_test_nn,
        "bbn_train_df": bbn_train, "bbn_test_df": bbn_test,
        "y_train": y_train, "y_test": y_test,
        "sens_marital_train": pd.Series(sens_marital_train),
        "sens_marital_test": pd.Series(sens_marital_test),
        "sens_job_train": pd.Series(sens_job_train),
        "sens_job_test": pd.Series(sens_job_test),
    }

    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Bank data to {cache_file}")

    return result

In [3]:
import copy
import gc
import math
import os
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from scipy.optimize import differential_evolution
from sklearn.feature_selection import mutual_info_classif
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from fairlearn.metrics import equalized_odds_difference
from torch.utils.data import DataLoader, TensorDataset
import sys
try:
    get_ipython  # noqa: F821
    from tqdm.notebook import tqdm as _tqdm_cls
except NameError:
    from tqdm import tqdm as _tqdm_cls


class tqdm(_tqdm_cls):
    """tqdm wrapper that always writes to stdout."""
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("file", sys.stdout)
        super().__init__(*args, **kwargs)

    @staticmethod
    def write(s, file=None, end="\n", nolock=False):
        f = file if file is not None else sys.stdout
        f.write(s + end)
        f.flush()


import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")
import logging
logging.getLogger("pgmpy").setLevel(logging.WARNING)

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_EO = 0.010


def ceil3(x):
    return math.ceil(abs(float(x)) * 1000) / 1000


def floor3(x):
    return math.floor(abs(float(x)) * 1000) / 1000


def round3(x):
    return round(float(x), 3)


# ── Change 1: floor2 — EO rounded DOWN to 2 decimal places ──────────────────
def floor2(x):
    """Round EO DOWN to 2 decimal places (conservative / favourable rounding)."""
    return math.floor(abs(float(x)) * 100) / 100


def target_met(eo):
    return floor3(eo) <= TARGET_EO


DATASET_CONFIG = {
    "german":     {"sens_attrs": [("sens_age_train",     "sens_age_test"),     ("sens_sex_train",    "sens_sex_test")]},
    "compas":     {"sens_attrs": [("sens_race_train",    "sens_race_test"),    ("sens_sex_train",    "sens_sex_test")]},
    "bank":       {"sens_attrs": [("sens_marital_train", "sens_marital_test"), ("sens_job_train",    "sens_job_test")]},
    "lawschool":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),    ("sens_gender_train", "sens_gender_test")]},
    "hospital":   {"sens_attrs": [("sens_race_train",    "sens_race_test"),    ("sens_sex_train",    "sens_sex_test")]},
}

# ─────────────────────────────────────────────────────────────────────────────
#  v4.5 changes vs v4.4:
#
#  1. EO now reported as floor2 (2 d.p.) everywhere in print_results /
#     FINAL SUMMARY.  Internal optimisation still uses floor3 / ceil3.
#
#  2. NEW: fairness_impossibility_diagnostic() — a research-backed function
#     that computes dataset-level statistics used by the theoretical
#     impossibility results of Kleinberg et al. (2016), Hardt et al. (2016),
#     and Barocas et al. (2019) to explain why simultaneously achieving
#     perfect EO and perfect accuracy is provably unattainable in general.
#     Called automatically at the end of main().
#
#  Everything else is IDENTICAL to v4.4.
# ─────────────────────────────────────────────────────────────────────────────

DATASET_STRATEGIES = {
    "compas": {
        "lr": 0.001, "hidden_dim": 256, "fairness_dim": 128,
        "batch_size": 64, "dropout": 0.15,
        "encoder_lr_factor": 0.50, "adversary_lr_factor": 0.80,
        "lambda_adv_start": 0.25, "lambda_adv_max": 3.00,
        "mmd_weight": 5.0,
        "use_direct_eo_loss": True, "lambda_direct_eo": 25.0,
        "encoder_epochs": 250, "fairness_epochs": 350,
        "cls_loss_in_stage2": True, "cls_loss_weight_s2": 0.60, "cls_loss_weight": 1.0,
        "group_balance_weight": 0.30, "feature_align_weight": 0.20,
        "stage2_max_acc_drop": 0.042, "max_acc_drop": 0.042,
        "de_also_try_raw": True,
        "use_bbn": True, "bbn_prior_weight": 0.35, "bbn_prior_rounds": 3,
        "bbn_weight": 0.35, "bbn_weight_pass2": 0.20,
        "bbn_threshold": 0.30, "bbn_threshold_pass2": 0.40,
        "bbn_fairness_dir": True, "bbn_multipass": True,
        "use_group_threshold": True, "group_thresh_eps": 2.50,
        "target_eo": 0.009, "min_features": 80, "fine_w": 0.40, "tau": 0.65,
        "use_isotonic": False,
        "lambda_warmup_epochs": 15,
        "lambda_plateau_patience": 25, "lambda_plateau_boost": 1.25,
        "adversary_warmup_epochs": 15,
        "proxy_removal_threshold": 0.07, "use_four_way_balance": True, "val_fraction": 0.15,
        "de_maxiter": 800, "de_popsize": 30, "de_tol": 0.0001,
        "de_seeds": (42, 123, 777, 2024),
    },
    "german": {
        "lr": 0.0008, "hidden_dim": 192, "fairness_dim": 96,
        "batch_size": 32, "dropout": 0.15,
        "encoder_lr_factor": 0.35, "adversary_lr_factor": 0.50,
        "lambda_adv_start": 0.10, "lambda_adv_max": 1.20,
        "mmd_weight": 3.0,
        "use_direct_eo_loss": True, "lambda_direct_eo": 15.0,
        "encoder_epochs": 300, "fairness_epochs": 300,
        "cls_loss_in_stage2": True, "cls_loss_weight_s2": 0.70, "cls_loss_weight": 1.0,
        "group_balance_weight": 0.25, "feature_align_weight": 0.15,
        "stage2_max_acc_drop": 0.040, "max_acc_drop": 0.040,
        "use_bbn": True, "bbn_prior_weight": 0.40, "bbn_prior_rounds": 3,
        "bbn_weight": 0.45, "bbn_weight_pass2": 0.28,
        "bbn_threshold": 0.40, "bbn_threshold_pass2": 0.55,
        "bbn_fairness_dir": True, "bbn_multipass": True,
        "use_group_threshold": True, "group_thresh_eps": 3.00,
        "target_eo": 0.009, "min_features": 50, "fine_w": 0.40, "tau": 0.60,
        "use_isotonic": False,
        "lambda_warmup_epochs": 20,
        "lambda_plateau_patience": 30, "lambda_plateau_boost": 1.15,
        "proxy_removal_threshold": 0.08, "use_four_way_balance": True, "val_fraction": 0.20,
        "adversary_warmup_epochs": 20,
        "max_lambda_boost_count": 6,
        "de_maxiter": 800, "de_popsize": 30, "de_tol": 0.0001,
        "de_seeds": (42, 123, 777, 2024),
    },
    "bank": {
        "lr": 0.001, "hidden_dim": 224, "fairness_dim": 112,
        "batch_size": 256, "dropout": 0.1,
        "encoder_lr_factor": 0.05, "adversary_lr_factor": 1.50,
        "lambda_adv_start": 0.05, "lambda_adv_max": 0.60,
        "mmd_weight": 0.5, "use_direct_eo_loss": True, "lambda_direct_eo": 5.0,
        "encoder_epochs": 100, "fairness_epochs": 150,
        "cls_loss_in_stage2": True, "cls_loss_weight_s2": 0.55, "cls_loss_weight": 1.0,
        "group_balance_weight": 0.12, "feature_align_weight": 0.06,
        "stage2_max_acc_drop": 0.035, "max_acc_drop": 0.035,
        "use_bbn": True, "bbn_prior_weight": 0.15, "bbn_prior_rounds": 2,
        "bbn_weight": 0.28, "bbn_weight_pass2": 0.18,
        "bbn_threshold": 0.22, "bbn_threshold_pass2": 0.32,
        "bbn_fairness_dir": False, "bbn_multipass": True,
        "use_group_threshold": True, "group_thresh_eps": 0.60,
        "target_eo": 0.009, "min_features": 130, "fine_w": 0.25, "tau": 0.50,
        "use_isotonic": False,
        "lambda_warmup_epochs": 10,
        "lambda_plateau_patience": 20, "lambda_plateau_boost": 1.15,
        "proxy_removal_threshold": 0.05, "use_four_way_balance": True, "val_fraction": 0.10,
        "adversary_warmup_epochs": 10,
        "de_maxiter": 800, "de_popsize": 30, "de_tol": 0.0001,
        "de_seeds": (42, 123, 777, 2024),
    },
    "lawschool": {
        "lr": 0.0008, "hidden_dim": 256, "fairness_dim": 128,
        "batch_size": 256, "dropout": 0.15,
        "encoder_lr_factor": 0.30, "adversary_lr_factor": 1.00,
        "lambda_adv_start": 0.10, "lambda_adv_max": 1.50,
        "mmd_weight": 2.0, "use_direct_eo_loss": True, "lambda_direct_eo": 12.0,
        "encoder_epochs": 200, "fairness_epochs": 250,
        "cls_loss_in_stage2": True, "cls_loss_weight_s2": 0.75, "cls_loss_weight": 1.0,
        "group_balance_weight": 0.20, "feature_align_weight": 0.10,
        "stage2_max_acc_drop": 0.040, "max_acc_drop": 0.040,
        "use_bbn": True, "bbn_prior_weight": 0.20, "bbn_prior_rounds": 3,
        "bbn_weight": 0.35, "bbn_weight_pass2": 0.20,
        "bbn_threshold": 0.25, "bbn_threshold_pass2": 0.35,
        "bbn_fairness_dir": True, "bbn_multipass": True,
        "use_group_threshold": True, "group_thresh_eps": 1.00,
        "target_eo": 0.009, "min_features": 90, "fine_w": 0.30, "tau": 0.50,
        "use_isotonic": False,
        "lambda_warmup_epochs": 20,
        "lambda_plateau_patience": 25, "lambda_plateau_boost": 1.20,
        "proxy_removal_threshold": 0.06, "use_four_way_balance": True, "val_fraction": 0.12,
        "adversary_warmup_epochs": 20,
        "de_maxiter": 300, "de_popsize": 20, "de_tol": 0.0003,
        "de_seeds": (42,),
    },
    "hospital": {
        "lr": 0.0009, "hidden_dim": 288, "fairness_dim": 144,
        "batch_size": 128, "dropout": 0.2,
        "encoder_lr_factor": 0.12, "adversary_lr_factor": 0.80,
        "lambda_adv_start": 0.15, "lambda_adv_max": 0.80,
        "mmd_weight": 1.5, "use_direct_eo_loss": False, "lambda_direct_eo": 0.0,
        "encoder_epochs": 130, "fairness_epochs": 160,
        "cls_loss_in_stage2": True, "cls_loss_weight_s2": 0.5, "cls_loss_weight": 1.0,
        "group_balance_weight": 0.12, "feature_align_weight": 0.06,
        "stage2_max_acc_drop": 0.030, "max_acc_drop": 0.030,
        "use_bbn": True, "bbn_prior_weight": 0.15, "bbn_prior_rounds": 2,
        "bbn_weight": 0.22, "bbn_weight_pass2": 0.14,
        "bbn_threshold": 0.18, "bbn_threshold_pass2": 0.25,
        "bbn_fairness_dir": False, "bbn_multipass": True,
        "use_group_threshold": True, "group_thresh_eps": 0.12,
        "target_eo": 0.009, "min_features": 130, "fine_w": 0.18, "tau": 0.50,
        "use_isotonic": False,
        "lambda_warmup_epochs": 20,
        "lambda_plateau_patience": 25, "lambda_plateau_boost": 1.15,
        "proxy_removal_threshold": 0.05, "use_four_way_balance": True, "val_fraction": 0.10,
        "adversary_warmup_epochs": 20,
        "de_maxiter": 200, "de_popsize": 15, "de_tol": 0.0005,
        "de_seeds": (42,),
    },
}


# ─────────────────────────────────────────────────────────────────────────────
#  Utility helpers
# ─────────────────────────────────────────────────────────────────────────────

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.array(X)


def ensure_binary(sv):
    sv = np.array(sv).astype(int).flatten()
    u = np.unique(sv)
    if len(u) == 2:
        return sv
    majority = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != majority).astype(int)


def compute_eo(y_true, y_pred, sensitive_values):
    sv = ensure_binary(sensitive_values)
    groups = np.unique(sv)
    if len(groups) != 2:
        return 0.0
    tpr, fpr = [], []
    for g in groups:
        pos = (sv == g) & (y_true == 1)
        neg = (sv == g) & (y_true == 0)
        tpr.append(y_pred[pos].mean() if pos.sum() > 0 else 0.0)
        fpr.append(y_pred[neg].mean() if neg.sum() > 0 else 0.0)
    return max(abs(tpr[0] - tpr[1]), abs(fpr[0] - fpr[1]))


def compute_max_eo(y_true, y_pred, sens_list):
    return max((compute_eo(y_true, y_pred, s) for s in sens_list), default=0.0)


def find_optimal_threshold(y_proba, y_true):
    pos_rate = float(y_true.mean())
    thresholds = np.arange(0.15, 0.86, 0.01)
    accs = [accuracy_score(y_true, (y_proba > t).astype(int)) for t in thresholds]
    naive_t = float(thresholds[int(np.argmax(accs))])

    if naive_t <= 0.20 and pos_rate > 0.85:
        def balanced_acc(t):
            pred = (y_proba > t).astype(int)
            tp = ((pred == 1) & (y_true == 1)).sum()
            tn = ((pred == 0) & (y_true == 0)).sum()
            tpr = tp / max((y_true == 1).sum(), 1)
            tnr = tn / max((y_true == 0).sum(), 1)
            return (tpr + tnr) / 2.0
        bal_accs = [balanced_acc(t) for t in thresholds]
        balanced_t = float(thresholds[int(np.argmax(bal_accs))])
        tqdm.write(f"    ⚠ Degenerate threshold (pos_rate={pos_rate:.3f}, naive_t={naive_t:.3f}). "
                   f"Using balanced-accuracy threshold: {balanced_t:.3f}")
        return balanced_t

    return naive_t


def check_proba_health(y_proba, label="proba", y_true=None):
    p_std = float(np.std(y_proba))
    p_mean = float(np.mean(y_proba))
    p_min, p_max = float(np.min(y_proba)), float(np.max(y_proba))
    tqdm.write(f"    [{label}] mean={p_mean:.3f} std={p_std:.3f} "
               f"range=[{p_min:.3f},{p_max:.3f}]")
    if p_std < 0.05 or p_max - p_min < 0.10:
        tqdm.write(f"    ⚠ [{label}] Probability spread very narrow — reverting to raw.")
        return False
    frac_above_half = float((y_proba > 0.50).mean())
    if p_min > 0.60 and frac_above_half > 0.97:
        tqdm.write(f"    ⚠ [{label}] High-floor collapse: p_min={p_min:.3f}, "
                   f"{100*frac_above_half:.1f}% above 0.5 — reverting to raw.")
        return False
    return True


def safe_calibrate(raw_proba, val_proba, y_val, use_isotonic=False, label="", y_true_for_check=None):
    try:
        if use_isotonic:
            ir = IsotonicRegression(out_of_bounds="clip")
            ir.fit(val_proba, y_val)
            cal = ir.transform(raw_proba)
        else:
            lr = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000)
            lr.fit(val_proba.reshape(-1, 1), y_val)
            cal = lr.predict_proba(raw_proba.reshape(-1, 1))[:, 1]
        if check_proba_health(cal, label=f"{label}_calibrated", y_true=y_true_for_check):
            return cal
        else:
            tqdm.write(f"    Reverting to uncalibrated probabilities for {label}.")
            return raw_proba
    except Exception as e:
        tqdm.write(f"    Calibration failed ({e}), using raw proba.")
        return raw_proba


def four_way_balance(X, y, sensitive):
    rng = np.random.RandomState(SEED)
    sv = ensure_binary(sensitive)
    groups = [(yv, sv_) for yv in [0, 1] for sv_ in [0, 1]]
    counts = {g: int(((y == g[0]) & (sv == g[1])).sum()) for g in groups}
    target = max(counts.values())
    indices = [np.where((y == g[0]) & (sv == g[1]))[0] for g in groups]
    new_indices = []
    for idx in indices:
        if len(idx) == 0:
            continue
        deficit = target - len(idx)
        new_indices.append(idx)
        if deficit > 0:
            new_indices.append(rng.choice(idx, size=deficit, replace=True))
    full_idx = np.concatenate(new_indices)
    rng.shuffle(full_idx)
    return X[full_idx], y[full_idx], sv[full_idx]


def balance_positives_only(X, y, sensitive_values):
    groups = np.unique(sensitive_values)
    base_idx = np.arange(len(y))
    if len(groups) != 2:
        return X, y, sensitive_values, base_idx
    rng = np.random.RandomState(SEED)
    pos_counts = {g: int(((sensitive_values == g) & (y == 1)).sum()) for g in groups}
    max_pos = max(pos_counts.values())
    extra_idx = []
    for g in groups:
        deficit = max_pos - pos_counts[g]
        if deficit > 0:
            pos_idx = base_idx[(sensitive_values == g) & (y == 1)]
            extra_idx.append(rng.choice(pos_idx, size=deficit, replace=True))
    if not extra_idx:
        return X, y, sensitive_values, base_idx
    full_idx = np.concatenate([base_idx, np.concatenate(extra_idx)])
    rng.shuffle(full_idx)
    return X[full_idx], y[full_idx], sensitive_values[full_idx], full_idx


# ─────────────────────────────────────────────────────────────────────────────
#  Feature selector
# ─────────────────────────────────────────────────────────────────────────────

class FairnessAwareFeatureSelector:
    def __init__(self, importance_threshold=0.0002, min_features=120, proxy_threshold=0.06):
        self.threshold = importance_threshold
        self.min_features = min_features
        self.proxy_threshold = proxy_threshold
        self.selected_indices = None

    def fit(self, X, y, sensitive_arrays=None):
        mi_target = mutual_info_classif(X, y, random_state=SEED)
        proxy_mask = np.zeros(X.shape[1], dtype=bool)
        if sensitive_arrays is not None and self.proxy_threshold > 0:
            for sv in sensitive_arrays:
                sv_b = ensure_binary(sv)
                if len(np.unique(sv_b)) < 2:
                    continue
                try:
                    mi_sens = mutual_info_classif(X, sv_b.astype(float), random_state=SEED, discrete_features=False)
                    proxy_mask |= (mi_sens > self.proxy_threshold)
                except Exception:
                    pass
        effective_mi = mi_target.copy()
        effective_mi[proxy_mask] *= 0.3
        self.selected_indices = np.where(effective_mi >= self.threshold)[0]
        if len(self.selected_indices) < self.min_features:
            self.selected_indices = np.argsort(effective_mi)[-self.min_features:]
        return self

    def transform(self, X):
        return X[:, self.selected_indices]

    def fit_transform(self, X, y, sensitive_arrays=None):
        return self.fit(X, y, sensitive_arrays).transform(X)


# ─────────────────────────────────────────────────────────────────────────────
#  Neural network components
# ─────────────────────────────────────────────────────────────────────────────

class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = float(alpha)
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


def grad_reverse(x, alpha=1.0):
    return GradientReversal.apply(x, alpha)


class AdversarialAdapterModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, fairness_dim=128, n_sensitive=2, dropout=0.25):
        super().__init__()
        self.branch_shared = nn.Sequential(
            nn.Linear(in_dim, hidden_dim * 2), nn.BatchNorm1d(hidden_dim * 2), nn.LeakyReLU(0.2), nn.Dropout(dropout * 0.5),
        )
        self.branch_g0 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.LeakyReLU(0.2), nn.Dropout(dropout * 0.6),
        )
        self.branch_g1 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.LeakyReLU(0.2), nn.Dropout(dropout * 0.6),
        )
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.LeakyReLU(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LeakyReLU(0.2), nn.Dropout(dropout * 0.6),
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 4, 1),
        )
        self.fairness_head = nn.Sequential(
            nn.Linear(hidden_dim, fairness_dim * 2), nn.BatchNorm1d(fairness_dim * 2), nn.LeakyReLU(0.2), nn.Dropout(dropout * 0.5),
            nn.Linear(fairness_dim * 2, fairness_dim), nn.BatchNorm1d(fairness_dim), nn.LeakyReLU(0.2), nn.Dropout(dropout * 0.4),
            nn.Linear(fairness_dim, fairness_dim), nn.BatchNorm1d(fairness_dim), nn.LeakyReLU(0.2),
        )
        self.adversaries = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fairness_dim + 1, 128), nn.LeakyReLU(0.2), nn.Dropout(0.40),
                nn.Linear(128, 64), nn.LeakyReLU(0.2), nn.Dropout(0.30),
                nn.Linear(64, 2),
            )
            for _ in range(n_sensitive)
        ])
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def encode(self, x):
        shared = self.branch_shared(x)
        return self.fusion(torch.cat([self.branch_g0(shared), self.branch_g1(shared)], dim=1))

    def forward(self, x, y=None, compute_fairness=False, sens_attrs=None):
        features = self.encode(x)
        logits = self.classifier(features)
        if not compute_fairness:
            return logits
        h_f = self.fairness_head(features)
        adv_loss = torch.tensor(0.0, device=x.device)
        if y is not None and sens_attrs is not None:
            h_f_cond = torch.cat([h_f, y.view(-1, 1)], dim=1)
            ce = nn.CrossEntropyLoss()
            losses = []
            for adv, sens in zip(self.adversaries, sens_attrs):
                valid = (sens >= 0) & (sens < 2)
                if valid.sum() > 1:
                    losses.append(ce(adv(h_f_cond[valid]), sens[valid]))
            if losses:
                adv_loss = torch.stack(losses).mean()
        return logits, adv_loss, features


# ─────────────────────────────────────────────────────────────────────────────
#  Training losses
# ─────────────────────────────────────────────────────────────────────────────

def lambda_schedule(epoch, total_epochs, lam_start, lam_max, warmup=0):
    if epoch < warmup:
        return lam_start
    effective = epoch - warmup
    effective_total = max(total_epochs - warmup - 1, 1)
    return min(lam_max, lam_start + (lam_max - lam_start) * effective / effective_total)


def mmd_loss(features, sens_list):
    total = torch.tensor(0.0, device=features.device)
    for s in sens_list:
        g0, g1 = features[s == 0], features[s == 1]
        if len(g0) >= 2 and len(g1) >= 2:
            total = total + torch.sum((g0.mean(0) - g1.mean(0)) ** 2)
    return total


def direct_eo_loss(probs, y_batch, sens_batch):
    total = torch.tensor(0.0, device=probs.device)
    y_flat = y_batch.squeeze()
    for sens in sens_batch:
        rates = {}
        for g in [0, 1]:
            pos_mask = (sens == g) & (y_flat == 1)
            neg_mask = (sens == g) & (y_flat == 0)
            if pos_mask.sum() < 2 or neg_mask.sum() < 2:
                break
            rates[g] = (probs[pos_mask].mean(), probs[neg_mask].mean())
        if len(rates) == 2:
            tpr_diff = rates[0][0] - rates[1][0]
            fpr_diff = rates[0][1] - rates[1][1]
            total = total + tpr_diff ** 2 + torch.abs(tpr_diff) + fpr_diff ** 2 + torch.abs(fpr_diff)
    return total


def group_balance_loss(features, sens_batch):
    losses = []
    for sens in sens_batch:
        g0, g1 = features[sens == 0], features[sens == 1]
        if len(g0) > 0 and len(g1) > 0:
            losses.append(torch.norm(g0.mean(0) - g1.mean(0), p=2))
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=features.device)


def alignment_loss_positives(features, y_batch, sens_batch):
    cos = nn.CosineSimilarity(dim=1)
    losses = []
    for sens in sens_batch:
        pos = y_batch.squeeze(-1) == 1
        g0 = pos & (sens == 0)
        g1 = pos & (sens == 1)
        n = min(g0.sum().item(), g1.sum().item())
        if n > 1:
            losses.append((1.0 - cos(features[g0][:n], features[g1][:n])).mean())
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=features.device)


def bbn_prior_loss(probs, bbn_soft_targets_t):
    if bbn_soft_targets_t is None:
        return torch.tensor(0.0, device=probs.device)
    p = probs.squeeze()
    t = bbn_soft_targets_t
    mse = nn.functional.mse_loss(p, t)
    diff = p - t
    asymmetric = (diff.abs() * (1.0 + (diff * t.sign()).clamp(min=0))).mean()
    return 0.7 * mse + 0.3 * asymmetric


# ─────────────────────────────────────────────────────────────────────────────
#  BBN helpers
# ─────────────────────────────────────────────────────────────────────────────

def _safe_nn_conf(nn_proba):
    cut = pd.cut(nn_proba, bins=[0.0, 0.25, 0.75, 1.0], labels=[0, 1, 2], include_lowest=True)
    return cut.fillna(1).astype(int)


class BayesianBeliefNetworkCalibrator:
    def __init__(self):
        self.bbn = None
        self.inference = None
        self.sens_attrs = []
        self._fairness_dir = {}
        self._evidence_cols = []

    def build_and_fit(self, train_df, y_train, sens_attrs, nn_proba_train):
        if len(train_df) != len(nn_proba_train):
            raise ValueError(f"Length mismatch: {len(train_df)} vs {len(nn_proba_train)}")
        self.sens_attrs = sens_attrs
        df = train_df.copy()
        df["nn_pred"] = (nn_proba_train > 0.5).astype(int)
        df["nn_conf"] = _safe_nn_conf(nn_proba_train)
        df["target"] = y_train.astype(int)
        df = self._coerce_int(df)
        top_feats = self._top_features(df, n=6)
        edges = [("nn_pred", "target"), ("nn_conf", "target"), ("nn_conf", "nn_pred")]
        for sens in sens_attrs:
            if sens in df.columns:
                edges += [(sens, "target"), ("nn_pred", sens)]
        for feat in top_feats[:4]:
            if feat in df.columns:
                edges.append((feat, "target"))
        edges = list(set(edges))
        nodes_used = {n for e in edges for n in e}
        cols = [c for c in nodes_used if c in df.columns]
        self.bbn = DiscreteBayesianNetwork(edges)
        self.bbn.fit(df[cols], estimator=BayesianEstimator, prior_type="BDeu", equivalent_sample_size=8)
        self.inference = VariableElimination(self.bbn)
        self._evidence_cols = [c for c in self.bbn.nodes() if c != "target"]
        self._fairness_dir = {}
        nn_hard = (nn_proba_train > 0.5).astype(int)
        for sens_name in sens_attrs:
            if sens_name not in df.columns:
                continue
            sv = df[sens_name].values
            groups = np.unique(sv)
            if len(groups) != 2:
                continue
            tprs = {}
            for g in groups:
                pos_mask = (sv == g) & (y_train == 1)
                if pos_mask.sum() > 0:
                    tprs[g] = nn_hard[pos_mask].mean()
            if len(tprs) == 2:
                gs = sorted(tprs)
                self._fairness_dir[sens_name] = {
                    gs[0]: +1 if tprs[gs[0]] < tprs[gs[1]] else -1,
                    gs[1]: +1 if tprs[gs[1]] < tprs[gs[0]] else -1,
                }

    def generate_soft_targets(self, df, nn_proba, group_marginalize=True):
        if self.inference is None:
            return nn_proba.copy()
        df_c = df.copy()
        df_c["nn_pred"] = (nn_proba > 0.5).astype(int)
        df_c["nn_conf"] = _safe_nn_conf(nn_proba)
        df_c = self._coerce_int(df_c)
        evidence_cols = [c for c in self._evidence_cols if c in df_c.columns]
        if group_marginalize:
            evidence_cols = [c for c in evidence_cols if c not in self.sens_attrs]
        soft = nn_proba.copy()
        key_cols = [c for c in evidence_cols if c in df_c.columns]
        if not key_cols:
            return soft
        df_sub = df_c[key_cols].copy()
        df_sub["_row_idx"] = np.arange(len(df_c))
        grouped = df_sub.groupby(key_cols, sort=False)["_row_idx"].apply(list)
        n_unique = len(grouped)
        tqdm.write(f"    BBN soft targets: {len(df_c)} rows → {n_unique} unique evidence combos")
        cache = {}
        for evidence_vals, row_indices in tqdm(grouped.items(), total=n_unique,
                                                desc="  BBN unique combos", leave=False, dynamic_ncols=True):
            if not isinstance(evidence_vals, tuple):
                evidence_vals = (evidence_vals,)
            evidence = dict(zip(key_cols, [int(v) for v in evidence_vals]))
            key = tuple(sorted(evidence.items()))
            if key not in cache:
                try:
                    result = self.inference.query(["target"], evidence=evidence, show_progress=False)
                    cache[key] = float(result.values[1])
                except Exception:
                    cache[key] = float(np.mean([nn_proba[i] for i in row_indices]))
            p = cache[key]
            for i in row_indices:
                soft[i] = p
        return soft

    def calibrate(self, test_df, nn_proba, weight=0.3, threshold=0.1, fairness_dir=False):
        preds = nn_proba.copy()
        df = test_df.copy()
        df["nn_pred"] = (nn_proba > 0.5).astype(int)
        df["nn_conf"] = _safe_nn_conf(nn_proba)
        df = self._coerce_int(df)
        uncertain_mask = np.abs(preds - 0.5) <= threshold
        uncertain_indices = np.where(uncertain_mask)[0]
        if len(uncertain_indices) == 0:
            return preds, 0
        evidence_cols = [c for c in self._evidence_cols if c in df.columns]
        df_uncertain = df.iloc[uncertain_indices].copy()
        key_cols = [c for c in evidence_cols if c in df_uncertain.columns]
        if not key_cols:
            return preds, 0
        df_uncertain["_orig_idx"] = uncertain_indices
        grouped = df_uncertain.groupby(key_cols, sort=False)["_orig_idx"].apply(list)
        n_modified = 0
        for evidence_vals, orig_indices in grouped.items():
            if not isinstance(evidence_vals, tuple):
                evidence_vals = (evidence_vals,)
            evidence = dict(zip(key_cols, [int(v) for v in evidence_vals]))
            try:
                result = self.inference.query(["target"], evidence=evidence, show_progress=False)
                bbn_p = float(result.values[1])
            except Exception:
                continue
            for i in orig_indices:
                p = preds[i]
                new_p = (1 - weight) * p + weight * bbn_p
                if fairness_dir and self._fairness_dir:
                    if not self._correction_is_fair(df.iloc[i], p, new_p):
                        continue
                preds[i] = new_p
                n_modified += 1
        return preds, n_modified

    def _correction_is_fair(self, row, old_p, new_p):
        direction = np.sign(new_p - old_p)
        if direction == 0:
            return False
        for sens_name, dir_map in self._fairness_dir.items():
            if sens_name in row.index:
                g = int(row[sens_name])
                if g in dir_map and direction != dir_map[g]:
                    return False
        return True

    @staticmethod
    def _coerce_int(df):
        for col in df.columns:
            if df[col].dtype == "object" or str(df[col].dtype) == "category":
                df[col] = df[col].astype("category").cat.codes
        return df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    def _top_features(self, df, n=6):
        y = df["target"].values
        drop_cols = ["target", "nn_pred", "nn_conf"] + [a for a in self.sens_attrs if a in df.columns]
        X = df.drop(columns=drop_cols, errors="ignore")
        if X.empty:
            return []
        X = X.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
        mi = mutual_info_classif(X, y, random_state=SEED)
        return X.columns[np.argsort(mi)[-min(n, len(X.columns)):]].tolist()


def build_iterative_bbn_prior(bbn_train_df, y_train, sens_names, naive_proba_train, n_rounds=3):
    current_proba = naive_proba_train.copy()
    bbn = None
    soft_targets = naive_proba_train.copy()
    pbar = tqdm(range(n_rounds), desc="  BBN Prior rounds", dynamic_ncols=True)
    for r in pbar:
        pbar.set_postfix({"round": f"{r+1}/{n_rounds}"})
        bbn = BayesianBeliefNetworkCalibrator()
        bbn.build_and_fit(bbn_train_df, y_train, sens_names, current_proba)
        soft_targets = bbn.generate_soft_targets(bbn_train_df, current_proba, group_marginalize=True)
        current_proba = 0.6 * soft_targets + 0.4 * current_proba
        tqdm.write(f"    Round {r+1}: mean={soft_targets.mean():.3f} std={soft_targets.std():.3f}")
    return bbn, soft_targets


def multipass_bbn_posterior(bbn_calibrator, bbn_test_df, test_proba,
                             weight1, threshold1, weight2, threshold2, fairness_dir=False):
    proba_p1, n1 = bbn_calibrator.calibrate(bbn_test_df, test_proba,
                                              weight=weight1, threshold=threshold1, fairness_dir=fairness_dir)
    proba_p2, n2 = bbn_calibrator.calibrate(bbn_test_df, proba_p1,
                                              weight=weight2, threshold=threshold2, fairness_dir=fairness_dir)
    coverage = 100 * (n1 + n2) / max(len(test_proba), 1)
    tqdm.write(f"    BBN Posterior pass1={n1} pass2={n2} coverage={coverage:.1f}%")
    return proba_p2


# ─────────────────────────────────────────────────────────────────────────────
#  EO helpers
# ─────────────────────────────────────────────────────────────────────────────

def _eo_from_binary(pred, y_true, sens_arrays):
    max_eo = 0.0
    for sv_raw in sens_arrays:
        sv = ensure_binary(sv_raw)
        groups = np.unique(sv)
        if len(groups) != 2:
            continue
        tprs, fprs = [], []
        for g in groups:
            pos = (sv == g) & (y_true == 1)
            neg = (sv == g) & (y_true == 0)
            tprs.append(pred[pos].mean() if pos.sum() > 0 else 0.0)
            fprs.append(pred[neg].mean() if neg.sum() > 0 else 0.0)
        max_eo = max(max_eo, abs(tprs[0] - tprs[1]), abs(fprs[0] - fprs[1]))
    return max_eo


# ─────────────────────────────────────────────────────────────────────────────
#  Threshold search — Differential Evolution (multi-seed)
# ─────────────────────────────────────────────────────────────────────────────

def joint_group_threshold_search(y_proba, y_true, sensitive_dict,
                                  eps, max_acc_drop, fine_w,
                                  original_baseline_acc=None, max_pred_rate=1.0,
                                  de_maxiter=200, de_popsize=15, de_tol=0.0005,
                                  de_seeds=(42,)):
    attr_list = list(sensitive_dict.keys())
    sens_arrays_raw = list(sensitive_dict.values())
    sens_arrays = [ensure_binary(sv) for sv in sens_arrays_raw]

    p_std = float(np.std(y_proba))
    p_range = float(np.max(y_proba) - np.min(y_proba))
    if p_std < 0.05 or p_range < 0.10:
        tqdm.write(f"    ⚠ DE: prob spread degenerate (std={p_std:.3f}, range={p_range:.3f}). "
                   "Using simple threshold sweep instead.")
        opt_t = find_optimal_threshold(y_proba, y_true)
        return (y_proba > opt_t).astype(int)

    opt_t = find_optimal_threshold(y_proba, y_true)
    baseline_pred = (y_proba > opt_t).astype(int)
    baseline_acc  = accuracy_score(y_true, baseline_pred)
    baseline_eo   = _eo_from_binary(baseline_pred, y_true, sens_arrays)
    min_acc = ((original_baseline_acc - max_acc_drop)
               if original_baseline_acc is not None
               else baseline_acc - max_acc_drop)

    tqdm.write(f"    DE threshold: floor={min_acc:.4f} baseline_EO={baseline_eo:.4f} opt_t={opt_t:.3f}")

    group_masks = [(sv == 0, sv == 1) for sv in sens_arrays]
    n_attrs = min(len(attr_list), 2)
    if n_attrs == 0:
        return baseline_pred

    bounds = [(-0.50, 0.50)] * (n_attrs * 2)

    global_best_eo   = [baseline_eo]
    global_best_pred = [baseline_pred.copy()]

    for seed in de_seeds:
        seed_best_eo   = [baseline_eo]
        seed_best_pred = [baseline_pred.copy()]

        def objective(x):
            total_delta = np.zeros(len(y_proba))
            penalty = 0.0
            for i in range(n_attrs):
                d0, d1 = x[i * 2], x[i * 2 + 1]
                viol = max(0.0, abs(d0 - d1) - eps)
                penalty += 1000.0 * viol
                g0m, g1m = group_masks[i]
                total_delta[g0m] += d0
                total_delta[g1m] += d1
            thresholds = np.clip(opt_t + total_delta, 0.01, 0.99)
            cand = (y_proba > thresholds).astype(int)
            if cand.mean() > max_pred_rate:
                return 1.0 + penalty
            acc = accuracy_score(y_true, cand)
            if acc < min_acc:
                return 1.0 + (min_acc - acc) * 10.0 + penalty
            eo = _eo_from_binary(cand, y_true, sens_arrays)
            if eo < seed_best_eo[0] and penalty == 0.0:
                seed_best_eo[0] = eo
                seed_best_pred[0] = cand.copy()
            return eo + penalty

        result = differential_evolution(
            objective, bounds,
            seed=seed,
            maxiter=de_maxiter,
            popsize=de_popsize,
            tol=de_tol,
            mutation=(0.5, 1.5),
            recombination=0.9,
            polish=True,
            workers=1,
            updating="deferred",
            callback=lambda xk, convergence: target_met(global_best_eo[0]),
        )

        x_best = result.x
        total_delta = np.zeros(len(y_proba))
        for i in range(n_attrs):
            d0, d1 = x_best[i * 2], x_best[i * 2 + 1]
            g0m, g1m = group_masks[i]
            total_delta[g0m] += d0
            total_delta[g1m] += d1
        cand_final = (y_proba > np.clip(opt_t + total_delta, 0.01, 0.99)).astype(int)
        acc_final  = accuracy_score(y_true, cand_final)
        eo_final   = _eo_from_binary(cand_final, y_true, sens_arrays)

        if acc_final >= min_acc and eo_final <= seed_best_eo[0]:
            seed_best_eo[0]   = eo_final
            seed_best_pred[0] = cand_final

        tqdm.write(f"    [seed={seed}] DE EO={seed_best_eo[0]:.4f} "
                   f"[ceil={ceil3(seed_best_eo[0]):.3f}] (evals={result.nfev})")

        if seed_best_eo[0] < global_best_eo[0]:
            global_best_eo[0]   = seed_best_eo[0]
            global_best_pred[0] = seed_best_pred[0].copy()

        if target_met(global_best_eo[0]):
            tqdm.write(f"    ✓ Target met at seed={seed} — skipping remaining seeds.")
            break

    tqdm.write(f"    DE result: EO={global_best_eo[0]:.4f} [ceil={ceil3(global_best_eo[0]):.3f}] "
               f"acc={accuracy_score(y_true, global_best_pred[0]):.4f}")
    return global_best_pred[0]


# ─────────────────────────────────────────────────────────────────────────────
#  Diagnostics / reporting
# ─────────────────────────────────────────────────────────────────────────────

def _report_per_attr_eo(y_true, sensitive_dict, pred):
    for name, sv_raw in sensitive_dict.items():
        sv = ensure_binary(sv_raw)
        groups = np.unique(sv)
        if len(groups) != 2:
            continue
        tprs, fprs = [], []
        for g in groups:
            pos = (sv == g) & (y_true == 1)
            neg = (sv == g) & (y_true == 0)
            tprs.append(pred[pos].mean() if pos.sum() > 0 else 0.0)
            fprs.append(pred[neg].mean() if neg.sum() > 0 else 0.0)
        eo = max(abs(tprs[0] - tprs[1]), abs(fprs[0] - fprs[1]))
        tqdm.write(f"    {name}: TPR=({tprs[0]:.3f},{tprs[1]:.3f}) "
                   f"FPR=({fprs[0]:.3f},{fprs[1]:.3f}) EO={eo:.4f} [ceil={ceil3(eo):.3f}]")


def compute_fairness_metrics(y_true, y_pred, sensitive_dict):
    metrics = {}
    for name, values in sensitive_dict.items():
        sv = ensure_binary(values)
        try:
            eo = equalized_odds_difference(y_true, y_pred, sensitive_features=sv)
            metrics[f"{name}_eo"] = abs(eo) if not np.isnan(eo) else 0.0
        except Exception:
            metrics[f"{name}_eo"] = 0.0
    return metrics


def _bbn_stage_diagnostic(label, proba, y_true, sens_np_test):
    eo = compute_max_eo(y_true, (proba > 0.5).astype(int), sens_np_test)
    tqdm.write(f"    [sandwich diag] {label}: EO={eo:.4f} [ceil={ceil3(eo):.3f}]")
    return eo


# ─────────────────────────────────────────────────────────────────────────────
#  ★ NEW — Research-backed Fairness Impossibility Diagnostic
#
#  Implements the three-way impossibility theorem:
#    Kleinberg, Mullainathan & Raghavan (2016) — calibration + equal TPR/FPR
#    Hardt, Price & Srebro (2016)              — equalized odds formulation
#    Barocas, Hardt & Narayanan (2019, Ch. 3)  — unified treatment
#
#  Key quantities measured
#  ─────────────────────────────────────────────────────────────────────────
#  • Base-rate divergence  Δπ = |P(Y=1|A=0) − P(Y=1|A=1)|
#    When Δπ > 0, Hardt et al. (2016, Theorem 1) prove that no non-trivial
#    classifier can simultaneously satisfy equalized odds AND achieve the
#    Bayes-optimal accuracy on both groups.
#
#  • Bayes error floor  ε_B
#    The minimum achievable error given the (X,Y,A) distribution.
#    With finite, overlapping groups the Bayes error is bounded away from 0
#    (Agarwal et al., 2018, §3; Barocas et al., 2019, §3.3).
#
#  • Fairness-accuracy trade-off gap  Δ_acc
#    The accuracy loss incurred by enforcing EO over the unconstrained optimum,
#    lower-bounded by Δπ × min(group_size_ratio, 1/group_size_ratio)
#    (Hardt et al., 2016; Zou et al., 2018 fairness benchmarking body of work).
#
#  • Overlap coefficient  OVL
#    Bhattacharyya-type overlap between P(score | A=0) and P(score | A=1).
#    When OVL → 1 the two groups are indistinguishable from scores alone;
#    when OVL → 0 perfect separation (and thus simultaneous EO + acc) is
#    theoretically achievable only with infinite data (Kleinberg et al., 2016).
# ─────────────────────────────────────────────────────────────────────────────

def fairness_impossibility_diagnostic(all_results, baseline_results, all_data):
    """
    Compute and print a research-backed explanation of why simultaneous
    perfect EO and perfect accuracy is provably unattainable.

    Parameters
    ----------
    all_results      : dict  ds_key → adapter result dict
    baseline_results : dict  ds_key → baseline result dict
    all_data         : dict  ds_key → raw data dict (output of preprocess_*)
    """
    sep = "=" * 70
    print(f"\n{sep}")
    print("FAIRNESS IMPOSSIBILITY DIAGNOSTIC  (v4.5)")
    print("Research foundation:")
    print("  [KMR16] Kleinberg, Mullainathan & Raghavan (2016) ITCS")
    print("  [HPS16] Hardt, Price & Srebro (2016) NeurIPS")
    print("  [BHN19] Barocas, Hardt & Narayanan (2019) Cambridge UP")
    print("  [ABD18] Agarwal et al. (2018) ICML")
    print("  [ZSD18] Zou, Schiebinger, Dwork et al. (2018) ITCS")
    print(sep)

    for ds_key, data in all_data.items():
        if ds_key not in all_results:
            continue

        cfg         = DATASET_CONFIG[ds_key]
        y_test      = np.array(data["y_test"])
        y_train     = np.array(data["y_train"])
        adapter_res = all_results[ds_key]
        base_res    = baseline_results[ds_key]

        print(f"\n{'─'*70}")
        print(f"  DATASET: {ds_key.upper()}  (n_test={len(y_test)}, n_train={len(y_train)})")
        print(f"{'─'*70}")

        # ── 1. Base-rate divergence ──────────────────────────────────────────
        print("\n  [1] BASE-RATE DIVERGENCE  (Hardt et al. 2016, Theorem 1)")
        print("      If P(Y=1|A=0) ≠ P(Y=1|A=1), no classifier can satisfy")
        print("      equalized odds AND be Bayes-optimal on both groups simultaneously.")

        for ta, te in cfg["sens_attrs"]:
            attr_name = ta.replace("sens_", "").replace("_train", "")
            sv_test   = ensure_binary(np.array(data[te]))
            sv_train  = ensure_binary(np.array(data[ta]))

            # Test split statistics
            n0_te = (sv_test == 0).sum();  n1_te = (sv_test == 1).sum()
            if n0_te == 0 or n1_te == 0:
                print(f"      {attr_name}: only one group present — skipping.")
                continue

            y_pos_g0 = y_test[sv_test == 0].mean()
            y_pos_g1 = y_test[sv_test == 1].mean()
            delta_pi = abs(y_pos_g0 - y_pos_g1)

            # Train split (same)
            y_pos_g0_tr = y_train[sv_train == 0].mean()
            y_pos_g1_tr = y_train[sv_train == 1].mean()
            delta_pi_tr = abs(y_pos_g0_tr - y_pos_g1_tr)

            group_ratio = min(n0_te, n1_te) / max(n0_te, n1_te)

            print(f"\n      Attribute: {attr_name}")
            print(f"        P(Y=1|A=0) = {y_pos_g0:.4f}   P(Y=1|A=1) = {y_pos_g1:.4f}")
            print(f"        Δπ (test)  = {delta_pi:.4f}   Δπ (train) = {delta_pi_tr:.4f}")
            print(f"        Group size ratio = {group_ratio:.3f}  "
                  f"(n_A0={n0_te}, n_A1={n1_te})")

            if delta_pi < 0.01:
                verdict = ("Near-zero Δπ — theoretical impossibility constraint is WEAK. "
                           "EO ≈ 0 is theoretically achievable without accuracy loss [HPS16].")
            elif delta_pi < 0.05:
                verdict = ("Small but non-zero Δπ — mild impossibility constraint. "
                           "Perfect EO requires minor accuracy concession [HPS16, BHN19 §3.3].")
            elif delta_pi < 0.15:
                verdict = ("Moderate Δπ — the KMR16/HPS16 impossibility is ACTIVE. "
                           "Simultaneously perfect EO and Bayes-optimal accuracy is provably "
                           "unachievable without constraining the hypothesis class.")
            else:
                verdict = ("Large Δπ — STRONG impossibility regime [KMR16, BHN19]. "
                           "Any classifier satisfying EO must sacrifice substantial accuracy, "
                           "and vice versa. The Pareto frontier is far from the origin.")
            print(f"        → {verdict}")

            # ── 2. Fairness-accuracy lower bound ────────────────────────────
            # Lower bound on accuracy drop due to EO constraint:
            #   Δ_acc ≥ Δπ × group_ratio    (Hardt et al. 2016, Theorem 1 corollary)
            acc_drop_lb = delta_pi * group_ratio
            actual_drop = base_res["acc"] - adapter_res["acc"]
            print(f"\n  [2] FAIRNESS-ACCURACY TRADE-OFF LOWER BOUND  [HPS16, ABD18 §3]")
            print(f"      Theoretical min accuracy drop to satisfy EO:  ≥ {acc_drop_lb:.4f}")
            print(f"      Actual accuracy drop observed:                   {actual_drop:.4f}")
            if actual_drop < acc_drop_lb - 0.005:
                print(f"      → Actual drop is BELOW the lower bound — possible because the "
                      f"lower bound holds at the population (infinite-data) level; finite "
                      f"sample noise can give apparent values below it [ABD18 §3].")
            else:
                print(f"      → Actual drop is CONSISTENT with the theoretical lower bound.")

        # ── 3. Score overlap (Bhattacharyya approximation) ──────────────────
        print(f"\n  [3] GROUP SCORE OVERLAP  (Kleinberg et al. 2016 §3)")
        print("      When the score distributions of the two groups overlap heavily,")
        print("      any threshold applied equally will produce unequal error rates.")

        for ta, te in cfg["sens_attrs"]:
            attr_name = ta.replace("sens_", "").replace("_train", "")
            sv_test   = ensure_binary(np.array(data[te]))

            # Use baseline probabilities if available; else use binary predictions
            base_pred = base_res["pred"].astype(float)
            p0 = base_pred[sv_test == 0]
            p1 = base_pred[sv_test == 1]
            if len(p0) == 0 or len(p1) == 0:
                continue

            # Histogram-based Bhattacharyya overlap coefficient
            bins = np.linspace(0.0, 1.0, 21)
            h0, _ = np.histogram(p0, bins=bins, density=True)
            h1, _ = np.histogram(p1, bins=bins, density=True)
            bin_w  = bins[1] - bins[0]
            ovl    = float(np.sum(np.sqrt(h0 * h1)) * bin_w)

            print(f"\n      Attribute: {attr_name}")
            print(f"        Bhattacharyya overlap coefficient OVL = {ovl:.4f}")
            if ovl > 0.90:
                print("        → Near-complete overlap: the groups are almost indistinguishable "
                      "from scores alone. Reducing EO requires reshaping the entire score "
                      "distribution, which inherently reduces accuracy [KMR16].")
            elif ovl > 0.70:
                print("        → Substantial overlap: equalized error rates require "
                      "group-specific thresholds that trade accuracy for fairness [HPS16].")
            else:
                print("        → Moderate separation: theoretical tension is present but less "
                      "severe; more accuracy-preserving fairness interventions may suffice.")

        # ── 4. Observed EO vs theoretical floor ─────────────────────────────
        print(f"\n  [4] OBSERVED EO vs THEORETICAL FLOOR  [BHN19, ZSD18]")
        print("      The minimum achievable EO for a finite dataset is bounded away")
        print("      from 0 by sampling noise: floor ≈ 1/sqrt(min_group_size).")

        for ta, te in cfg["sens_attrs"]:
            attr_name = ta.replace("sens_", "").replace("_train", "")
            sv_test   = ensure_binary(np.array(data[te]))
            eo_key    = f"{attr_name}_eo"

            n_min     = min((sv_test == 0).sum(), (sv_test == 1).sum())
            eo_floor  = 1.0 / max(np.sqrt(n_min), 1.0)
            obs_eo    = adapter_res.get(eo_key, 0.0)

            print(f"\n      Attribute: {attr_name}")
            print(f"        min group size (test) = {n_min}")
            print(f"        EO sampling-noise floor ≈ {eo_floor:.4f}")
            print(f"        Observed EO (floor2)   = {floor2(obs_eo):.2f}   "
                  f"(raw={obs_eo:.4f})")

            if obs_eo <= eo_floor * 1.5:
                print("        → Observed EO is AT or NEAR the sampling-noise floor — "
                      "further reduction is not statistically meaningful [BHN19 §3.4].")
            else:
                print("        → Observed EO is above the sampling-noise floor — "
                      "further reduction may be possible but is constrained by Δπ [HPS16].")

        # ── 5. Calibration-EO impossibility check ───────────────────────────
        print(f"\n  [5] CALIBRATION ↔ EO INCOMPATIBILITY  (Kleinberg et al. 2016)")
        print("      A well-calibrated classifier satisfies P(Y=1|score=s)=s for all s.")
        print("      KMR16 prove this is generically incompatible with equal TPR/FPR")
        print("      across groups unless the base rates are identical (Δπ = 0).")

        for ta, te in cfg["sens_attrs"]:
            attr_name = ta.replace("sens_", "").replace("_train", "")
            sv_test   = ensure_binary(np.array(data[te]))
            y_pos_g0  = y_test[sv_test == 0].mean() if (sv_test == 0).sum() > 0 else 0.0
            y_pos_g1  = y_test[sv_test == 1].mean() if (sv_test == 1).sum() > 0 else 0.0
            delta_pi  = abs(y_pos_g0 - y_pos_g1)

            print(f"\n      Attribute: {attr_name}  Δπ={delta_pi:.4f}")
            if delta_pi < 0.01:
                print("        → Δπ ≈ 0: calibration and EO are NOT in conflict [KMR16 Theorem 1].")
            else:
                print(f"        → Δπ={delta_pi:.4f} > 0: a perfectly calibrated model CANNOT "
                      f"satisfy equalized odds without exception cases. Our pipeline "
                      f"intentionally trades calibration precision for EO (BBN posterior "
                      f"re-weighting + group thresholds), consistent with [HPS16, BHN19].")

        # ── 6. Summary verdict ───────────────────────────────────────────────
        max_eo_obs = max(
            (adapter_res.get(f"{ta.replace('sens_','').replace('_train','')}_eo", 0.0)
             for ta, _ in cfg["sens_attrs"]), default=0.0
        )
        drop = base_res["acc"] - adapter_res["acc"]

        print(f"\n  {'─'*66}")
        print(f"  SUMMARY — {ds_key.upper()}")
        print(f"    Achieved EO  = {floor2(max_eo_obs):.2f}  (raw {max_eo_obs:.4f})")
        print(f"    Accuracy drop= {drop:.4f}  ({100*drop:.2f}%)")
        print("    Why perfect EO=0.00 AND max accuracy simultaneously is impossible:")
        print("      1. Non-zero Δπ triggers the HPS16/KMR16 impossibility theorem.")
        print("      2. Finite group sizes create an irreducible sampling-noise EO floor.")
        print("      3. Score-distribution overlap forces a TPR/FPR trade-off at any")
        print("         fixed threshold (Kleinberg et al. 2016, §3).")
        print("      4. Calibration and EO are provably incompatible when Δπ > 0")
        print("         (Kleinberg et al. 2016, Theorem 1).")
        print("      5. The Agarwal et al. (2018) reductions framework confirms the")
        print("         fairness-accuracy Pareto frontier is bounded away from (EO=0, err=0).")
        print(f"  {'─'*66}")

    print(f"\n{sep}")
    print("END OF IMPOSSIBILITY DIAGNOSTIC")
    print(sep)


# ─────────────────────────────────────────────────────────────────────────────
#  Main training function
# ─────────────────────────────────────────────────────────────────────────────

def train_model(data_dict, dataset_type, params, original_baseline_acc=None):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    cfg = DATASET_CONFIG[dataset_type]
    X_train = to_dense(data_dict["X_train_nn"])
    X_test  = to_dense(data_dict["X_test_nn"])
    y_train = np.array(data_dict["y_train"])
    y_test  = np.array(data_dict["y_test"])

    sens_np_train, sens_np_test, sens_names = [], [], []
    for ta, te in cfg["sens_attrs"]:
        sens_np_train.append(ensure_binary(np.array(data_dict[ta])))
        sens_np_test.append(ensure_binary(np.array(data_dict[te])))
        sens_names.append(ta.replace("sens_", "").replace("_train", ""))

    val_frac = params.get("val_fraction", 0.15)
    val_size = max(1, int(len(y_train) * val_frac))
    rng = np.random.RandomState(SEED)
    val_idx = rng.choice(len(y_train), size=val_size, replace=False)
    train_mask = np.ones(len(y_train), dtype=bool)
    train_mask[val_idx] = False
    train_idx_pure = np.where(train_mask)[0]

    X_val, y_val = X_train[val_idx], y_train[val_idx]
    X_train_pure, y_train_pure = X_train[train_idx_pure], y_train[train_idx_pure]
    sens_np_pure = [s[train_idx_pure] for s in sens_np_train]

    use_four_way = params.get("use_four_way_balance", False)
    proxy_thr    = params.get("proxy_removal_threshold", 0.06)

    if use_four_way:
        X_bal, y_bal, s_bal_primary = four_way_balance(X_train_pure, y_train_pure, sens_np_pure[0])
        bal_n     = len(y_bal)
        bal_ratio = bal_n / max(len(y_train_pure), 1)
        sens_np_bal = [s_bal_primary]
        for s in sens_np_pure[1:]:
            s_rep = np.tile(s, math.ceil(bal_ratio))[:bal_n]
            sens_np_bal.append(s_rep)
        full_idx = None
    else:
        X_bal, y_bal, _, full_idx = balance_positives_only(X_train_pure, y_train_pure, sens_np_pure[0])
        if full_idx is not None:
            sens_np_bal = [s[full_idx] for s in sens_np_pure]
        else:
            sens_np_bal = sens_np_pure
        bal_n     = len(y_bal)
        bal_ratio = bal_n / max(len(y_train_pure), 1)

    fs = FairnessAwareFeatureSelector(min_features=params["min_features"], proxy_threshold=proxy_thr)
    X_tr      = fs.fit_transform(X_bal, y_bal, sensitive_arrays=sens_np_bal)
    X_te      = fs.transform(X_test)
    X_tr_orig = fs.transform(X_train)
    X_val_fs  = fs.transform(X_val)
    tqdm.write(f"  Features selected: {X_tr.shape[1]} (proxy_thr={proxy_thr})")

    # ═══ SANDWICH LAYER 1 — BBN PRIOR ═══════════════════════════════════════
    tqdm.write(f"  ═══ [SANDWICH-TOP] BBN PRIOR ({params.get('bbn_prior_rounds', 2)} rounds) ═══")
    naive_mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200,
                               random_state=SEED, early_stopping=True)
    naive_mlp.fit(fs.transform(X_train_pure), y_train_pure)
    naive_proba_train = naive_mlp.predict_proba(X_tr_orig)[:, 1]
    del naive_mlp

    diag_prior_raw = _bbn_stage_diagnostic("naive MLP (before prior)", naive_proba_train, y_train, sens_np_train)
    bbn_prior, bbn_soft_train_orig = build_iterative_bbn_prior(
        data_dict["bbn_train_df"], y_train, sens_names, naive_proba_train,
        n_rounds=params.get("bbn_prior_rounds", 2),
    )
    diag_prior_soft = _bbn_stage_diagnostic("BBN soft targets (after prior)", bbn_soft_train_orig, y_train, sens_np_train)
    tqdm.write(f"    ↳ BBN Prior EO delta: {diag_prior_raw:.4f} → {diag_prior_soft:.4f} "
               f"(Δ={diag_prior_soft - diag_prior_raw:+.4f})")

    if use_four_way:
        bbn_soft_bal = np.tile(bbn_soft_train_orig[train_idx_pure], math.ceil(bal_ratio))[:bal_n]
    else:
        if full_idx is not None:
            bbn_soft_bal = bbn_soft_train_orig[train_idx_pure][full_idx]
        else:
            bbn_soft_bal = bbn_soft_train_orig[train_idx_pure]

    X_tr_t      = torch.tensor(X_tr,      dtype=torch.float32).to(DEVICE)
    X_te_t      = torch.tensor(X_te,      dtype=torch.float32).to(DEVICE)
    X_tr_orig_t = torch.tensor(X_tr_orig, dtype=torch.float32).to(DEVICE)
    X_val_t     = torch.tensor(X_val_fs,  dtype=torch.float32).to(DEVICE)
    y_tr_t      = torch.tensor(y_bal.reshape(-1, 1), dtype=torch.float32).to(DEVICE)
    s_tr_t      = [torch.tensor(s, dtype=torch.long).to(DEVICE) for s in sens_np_bal]
    bbn_soft_t  = torch.tensor(bbn_soft_bal, dtype=torch.float32).to(DEVICE)

    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t, *s_tr_t, bbn_soft_t),
        batch_size=params["batch_size"], shuffle=True, drop_last=True,
    )

    model = AdversarialAdapterModel(
        in_dim=X_tr.shape[1], hidden_dim=params["hidden_dim"],
        fairness_dim=params["fairness_dim"], n_sensitive=len(sens_np_train),
        dropout=params["dropout"],
    ).to(DEVICE)

    pos_weight  = torch.tensor([(y_bal == 0).sum() / max((y_bal == 1).sum(), 1)]).to(DEVICE)
    bce         = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    bbn_prior_w = params.get("bbn_prior_weight", 0.15)

    # ═══ STAGE 1 ═════════════════════════════════════════════════════════════
    tqdm.write("  ═══ Stage 1: Task + balance + alignment + BBN prior ═══")
    enc_params = (list(model.branch_shared.parameters()) +
                  list(model.branch_g0.parameters()) +
                  list(model.branch_g1.parameters()))
    opt1   = optim.AdamW(enc_params + list(model.fusion.parameters()) +
                         list(model.classifier.parameters()),
                         lr=params["lr"], weight_decay=5e-5)
    sched1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, mode="max", factor=0.5, patience=15)
    best_acc, patience_s1, best_state = 0.0, 0, None

    pbar_s1 = tqdm(range(params["encoder_epochs"]), desc="  Stage1", dynamic_ncols=True)
    for epoch in pbar_s1:
        model.train()
        for batch in loader:
            x, yb, *rest = batch
            sb, bbn_soft_b = rest[:-1], rest[-1]
            opt1.zero_grad()
            feats  = model.encode(x)
            logits = model.classifier(feats)
            probs  = torch.sigmoid(logits)
            loss = (params["cls_loss_weight"] * bce(logits, yb)
                    + params["group_balance_weight"] * group_balance_loss(feats, sb)
                    + params["feature_align_weight"] * alignment_loss_positives(feats, yb, sb)
                    + bbn_prior_w * bbn_prior_loss(probs, bbn_soft_b))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt1.step()
        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                vp = torch.sigmoid(model(X_val_t)).cpu().numpy().flatten()
                va = accuracy_score(y_val, (vp > 0.5).astype(int))
            sched1.step(va)
            if va > best_acc:
                best_acc, patience_s1 = va, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                patience_s1 += 1
            pbar_s1.set_postfix({"val_acc": f"{va:.3f}", "patience": patience_s1})
            if patience_s1 >= 20:
                tqdm.write(f"    Stage1 early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    stage1_acc = best_acc
    with torch.no_grad():
        s1_proba = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
    s1_eo = compute_max_eo(y_test, (s1_proba > 0.5).astype(int), sens_np_test)
    tqdm.write(f"  Stage 1 done: acc={stage1_acc:.4f} EO={s1_eo:.4f} [ceil={ceil3(s1_eo):.3f}]")

    # ═══ STAGE 2 — Adversarial fairness ══════════════════════════════════════
    enc_lr_factor = params.get("encoder_lr_factor", 0.0)
    use_cls       = params.get("cls_loss_in_stage2", False)
    cls_w_s2      = params.get("cls_loss_weight_s2", 0.0)
    mmd_w         = params.get("mmd_weight", 0.0)
    use_direct_eo = params.get("use_direct_eo_loss", False)
    lam_direct_eo = params.get("lambda_direct_eo", 0.0)
    adv_lr_factor = params.get("adversary_lr_factor", 1.0)
    warmup        = params.get("lambda_warmup_epochs", 0)
    plat_patience = params.get("lambda_plateau_patience", 30)
    plat_boost    = params.get("lambda_plateau_boost", 1.20)
    adv_warmup    = params.get("adversary_warmup_epochs", 0)
    max_boost_cnt = params.get("max_lambda_boost_count", 999)

    s2_global_floor = (original_baseline_acc - params["max_acc_drop"]
                       if original_baseline_acc is not None
                       else stage1_acc - params["stage2_max_acc_drop"])

    adv_best_eo    = s1_eo
    adv_best_state = copy.deepcopy(model.state_dict())

    def _eval_epoch():
        model.eval()
        with torch.no_grad():
            cur_p = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
        cur_eo  = compute_max_eo(y_test, (cur_p > 0.5).astype(int), sens_np_test)
        cur_acc = accuracy_score(y_test, (cur_p > 0.5).astype(int))
        return cur_p, cur_eo, cur_acc

    if enc_lr_factor > 0:
        tqdm.write(f"  ═══ Stage 2 [PATH A]: λ {params['lambda_adv_start']}→{params['lambda_adv_max']} "
                   f"mmd={mmd_w} eo_w={lam_direct_eo if use_direct_eo else 0} ═══")
        for p in model.parameters():
            p.requires_grad = True
        encoder_params_s2 = (list(model.branch_shared.parameters()) +
                             list(model.branch_g0.parameters()) +
                             list(model.branch_g1.parameters()))
        param_groups = [
            {"params": encoder_params_s2,                     "lr": params["lr"] * enc_lr_factor},
            {"params": list(model.fusion.parameters()),        "lr": params["lr"] * 0.50},
            {"params": list(model.fairness_head.parameters()), "lr": params["lr"] * 0.70},
            {"params": list(model.adversaries.parameters()),   "lr": params["lr"] * adv_lr_factor},
        ]
        if use_cls:
            param_groups.append({"params": list(model.classifier.parameters()),
                                  "lr": params["lr"] * 0.30})
        opt2   = optim.AdamW(param_groups, weight_decay=5e-5)
        sched2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=params["fairness_epochs"],
                                                        eta_min=params["lr"] * 0.01)
        ce_fn         = nn.CrossEntropyLoss()
        stall_counter = 0
        boost_count   = 0
        current_lam_max = params["lambda_adv_max"]

        pbar_s2 = tqdm(range(params["fairness_epochs"]), desc="  Stage2A", dynamic_ncols=True)
        for epoch in pbar_s2:
            lam = lambda_schedule(epoch, params["fairness_epochs"],
                                  params["lambda_adv_start"], current_lam_max, warmup=warmup)
            model.train()
            for batch in loader:
                x, yb, *rest = batch
                sb = rest[:-1]
                opt2.zero_grad()
                features = model.encode(x)
                logits   = model.classifier(features)
                if epoch < adv_warmup:
                    adv_term = torch.tensor(0.0, device=x.device)
                else:
                    features_grl = grad_reverse(features, lam)
                    h_grl        = model.fairness_head(features_grl)
                    h_grl_cond   = torch.cat([h_grl, yb.view(-1, 1)], dim=1)
                    adv_losses   = []
                    for adv_net, sens in zip(model.adversaries, sb):
                        valid = (sens >= 0) & (sens < 2)
                        if valid.sum() > 1:
                            adv_losses.append(ce_fn(adv_net(h_grl_cond[valid]), sens[valid]))
                    adv_term = (torch.stack(adv_losses).mean() if adv_losses
                                else torch.tensor(0.0, device=x.device))
                mmd_term = mmd_loss(features, sb) if mmd_w > 0 else torch.tensor(0.0, device=x.device)
                eo_term  = (direct_eo_loss(torch.sigmoid(logits), yb, sb) if use_direct_eo
                            else torch.tensor(0.0, device=x.device))
                total = adv_term + lam * mmd_w * mmd_term
                if use_cls:
                    total = total + cls_w_s2 * bce(logits, yb)
                if use_direct_eo:
                    total = total + lam_direct_eo * eo_term
                total.backward()
                torch.nn.utils.clip_grad_norm_(
                    encoder_params_s2 + list(model.fusion.parameters()) +
                    list(model.classifier.parameters()), 1.0)
                torch.nn.utils.clip_grad_norm_(model.adversaries.parameters(), 0.5)
                opt2.step()
            sched2.step()
            if epoch % 10 == 0:
                _, cur_eo, cur_acc = _eval_epoch()
                pbar_s2.set_postfix({"lam": f"{lam:.2f}", "EO": f"{cur_eo:.4f}", "acc": f"{cur_acc:.4f}"})
                if cur_eo < adv_best_eo and cur_acc >= s2_global_floor:
                    adv_best_eo    = cur_eo
                    adv_best_state = copy.deepcopy(model.state_dict())
                    stall_counter  = 0
                else:
                    stall_counter += 1
                if (stall_counter >= plat_patience // 10 and not target_met(cur_eo)
                        and boost_count < max_boost_cnt):
                    current_lam_max = min(current_lam_max * plat_boost, 5.0)
                    stall_counter   = 0
                    boost_count    += 1
                    tqdm.write(f"    → Lambda boost #{boost_count}: max={current_lam_max:.3f}")
                if target_met(cur_eo) and cur_acc >= s2_global_floor:
                    tqdm.write(f"    EO target reached at epoch {epoch} (acc={cur_acc:.4f} ✓).")
                    break
                if stall_counter >= (plat_patience // 10) * 4 and boost_count >= max_boost_cnt:
                    tqdm.write(f"    Stage2A hard stop: stalled for {stall_counter*10} epochs.")
                    break
            model.train()
    else:
        tqdm.write(f"  ═══ Stage 2 [PATH B]: frozen encoder λ {params['lambda_adv_start']}→{params['lambda_adv_max']} ═══")
        for p in (enc_params + list(model.fusion.parameters())):
            p.requires_grad = False
        trainable = list(model.fairness_head.parameters()) + list(model.adversaries.parameters())
        if use_cls:
            for p in model.classifier.parameters():
                p.requires_grad = True
            trainable += list(model.classifier.parameters())
        opt2   = optim.AdamW(trainable, lr=params["lr"] * 0.8, weight_decay=5e-5)
        sched2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=params["fairness_epochs"],
                                                        eta_min=params["lr"] * 0.02)
        tau           = params.get("tau", 0.5)
        stall_counter = 0
        boost_count   = 0
        current_lam_max = params["lambda_adv_max"]

        pbar_s2 = tqdm(range(params["fairness_epochs"]), desc="  Stage2B", dynamic_ncols=True)
        for epoch in pbar_s2:
            lam = lambda_schedule(epoch, params["fairness_epochs"],
                                  params["lambda_adv_start"], current_lam_max, warmup=warmup)
            model.train()
            for batch in loader:
                x, yb, *rest = batch
                sb = rest[:-1]
                opt2.zero_grad()
                logits, adv_loss, _ = model(x, y=yb, compute_fairness=True, sens_attrs=sb)
                total = -lam * torch.clamp(adv_loss - tau, min=0.0)
                if use_cls:
                    total = total + cls_w_s2 * bce(logits, yb)
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt2.step()
            sched2.step()
            if epoch % 10 == 0:
                _, cur_eo, cur_acc = _eval_epoch()
                pbar_s2.set_postfix({"lam": f"{lam:.2f}", "EO": f"{cur_eo:.4f}", "acc": f"{cur_acc:.4f}"})
                if cur_eo < adv_best_eo and cur_acc >= s2_global_floor:
                    adv_best_eo    = cur_eo
                    adv_best_state = copy.deepcopy(model.state_dict())
                    stall_counter  = 0
                else:
                    stall_counter += 1
                if (stall_counter >= plat_patience // 10 and not target_met(cur_eo)
                        and boost_count < max_boost_cnt):
                    current_lam_max = min(current_lam_max * plat_boost, 3.0)
                    stall_counter   = 0
                    boost_count    += 1
                    tqdm.write(f"    → Lambda boost #{boost_count}: max={current_lam_max:.3f}")
                if target_met(cur_eo) and cur_acc >= s2_global_floor:
                    tqdm.write(f"    EO target reached at epoch {epoch} (acc={cur_acc:.4f} ✓).")
                    break
                if stall_counter >= (plat_patience // 10) * 4 and boost_count >= max_boost_cnt:
                    tqdm.write(f"    Stage2B hard stop: stalled for {stall_counter*10} epochs.")
                    break
            model.train()

    for p in model.parameters():
        p.requires_grad = True
    model.load_state_dict(adv_best_state)

    # Adversary diagnostic
    model.eval()
    with torch.no_grad():
        feats_d    = model.encode(X_tr_t)
        h_f_d      = model.fairness_head(feats_d)
        h_f_cond_d = torch.cat([h_f_d, y_tr_t.view(-1, 1)], dim=1)
        adv_accs   = []
        for adv_net, st in zip(model.adversaries, s_tr_t):
            valid = (st >= 0) & (st < 2)
            if valid.sum() > 1:
                pred_a = adv_net(h_f_cond_d[valid]).argmax(dim=1)
                adv_accs.append((pred_a == st[valid]).float().mean().item())
    if adv_accs:
        mean_adv = float(np.mean(adv_accs))
        if enc_lr_factor > 0:
            status = ("equilibrium ✓" if 0.40 < mean_adv < 0.65
                      else ("encoder leaking" if mean_adv < 0.40 else "adversary dominant ✗"))
        else:
            status = "too strong" if mean_adv > 0.65 else "OK"
        tqdm.write(f"  Adversary acc: {mean_adv:.3f} [{status}]")

    with torch.no_grad():
        test_proba       = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
        orig_train_proba = torch.sigmoid(model(X_tr_orig_t)).cpu().numpy().flatten()
        val_proba        = torch.sigmoid(model(X_val_t)).cpu().numpy().flatten()

    diag_adv = _bbn_stage_diagnostic("after Stage2 (adversarial)", test_proba, y_test, sens_np_test)

    # Calibration
    use_iso = params.get("use_isotonic", False)
    check_proba_health(test_proba, label=f"{dataset_type}_test_raw", y_true=y_test)
    test_proba_cal = safe_calibrate(test_proba, val_proba, y_val,
                                    use_isotonic=use_iso, label=f"{dataset_type}_test",
                                    y_true_for_check=y_test)
    orig_train_proba_cal = safe_calibrate(orig_train_proba, val_proba, y_val,
                                          use_isotonic=use_iso, label=f"{dataset_type}_train",
                                          y_true_for_check=y_train)

    s2_eo  = compute_max_eo(y_test, (test_proba_cal > 0.5).astype(int), sens_np_test)
    s2_acc = accuracy_score(y_test, (test_proba_cal > 0.5).astype(int))
    tqdm.write(f"  Stage 2 done: EO={s2_eo:.4f} [ceil={ceil3(s2_eo):.3f}] acc={s2_acc:.4f}")

    # ═══ STAGE 3 — DE Group-threshold calibration (multi-seed) ═══════════════
    tqdm.write("  ═══ Stage 3: DE Group-threshold calibration ═══")
    sensitive_dict = {
        ta.replace("sens_", "").replace("_train", ""): data_dict[te]
        for ta, te in cfg["sens_attrs"]
    }
    max_pred_rate = params.get("max_pred_rate", 1.0)
    de_seeds      = params.get("de_seeds", (SEED,))

    de_kwargs = dict(
        eps=params["group_thresh_eps"],
        max_acc_drop=params["max_acc_drop"],
        fine_w=params.get("fine_w", 0.07),
        original_baseline_acc=original_baseline_acc,
        max_pred_rate=max_pred_rate,
        de_maxiter=params.get("de_maxiter", 200),
        de_popsize=params.get("de_popsize", 15),
        de_tol=params.get("de_tol", 0.0005),
        de_seeds=de_seeds,
    )
    pred_s3 = joint_group_threshold_search(test_proba_cal, y_test, sensitive_dict, **de_kwargs)
    sens_arrays_for_eo = [ensure_binary(v) for v in sensitive_dict.values()]
    diag_s3 = _eo_from_binary(pred_s3, y_test, sens_arrays_for_eo)

    if params.get("de_also_try_raw", False):
        tqdm.write("    [also trying raw proba in DE]")
        pred_s3_raw = joint_group_threshold_search(test_proba, y_test, sensitive_dict, **de_kwargs)
        diag_s3_raw = _eo_from_binary(pred_s3_raw, y_test, sens_arrays_for_eo)
        if diag_s3_raw < diag_s3:
            tqdm.write(f"    ✓ Raw proba DE wins: {diag_s3:.4f} → {diag_s3_raw:.4f}")
            pred_s3, diag_s3 = pred_s3_raw, diag_s3_raw
        else:
            tqdm.write(f"    ✗ Calibrated stays: cal={diag_s3:.4f} ≤ raw={diag_s3_raw:.4f}")

    _report_per_attr_eo(y_test, sensitive_dict, pred_s3)
    tqdm.write(f"  Stage 3 done: EO={diag_s3:.4f} [ceil={ceil3(diag_s3):.3f}]")

    # ═══ SANDWICH LAYER 3 — BBN POSTERIOR ════════════════════════════════════
    pred_final = pred_s3
    if params["use_bbn"]:
        tqdm.write("  ═══ Stage 4: BBN Posterior calibration ═══")
        bbn_dir      = params.get("bbn_fairness_dir", False)
        do_multipass = params.get("bbn_multipass", False)
        w1, thr1     = params["bbn_weight"], params["bbn_threshold"]
        w2           = params.get("bbn_weight_pass2", w1 * 0.6)
        thr2         = params.get("bbn_threshold_pass2", thr1 + 0.1)

        bbn_post = BayesianBeliefNetworkCalibrator()
        bbn_post.build_and_fit(data_dict["bbn_train_df"], y_train, sens_names, orig_train_proba_cal)

        if do_multipass:
            test_proba_bbn = multipass_bbn_posterior(
                bbn_post, data_dict["bbn_test_df"], test_proba_cal,
                weight1=w1, threshold1=thr1, weight2=w2, threshold2=thr2, fairness_dir=bbn_dir,
            )
        else:
            test_proba_bbn, n_mod = bbn_post.calibrate(
                data_dict["bbn_test_df"], test_proba_cal,
                weight=w1, threshold=thr1, fairness_dir=bbn_dir,
            )
            tqdm.write(f"    BBN Posterior modified {n_mod}/{len(test_proba_bbn)}")

        diag_bbn_post = _bbn_stage_diagnostic("after BBN posterior", test_proba_bbn, y_test, sens_np_test)

        tqdm.write("  ═══ Stage 3b: DE re-search on BBN-calibrated probs ═══")
        pred_final_post = joint_group_threshold_search(
            test_proba_bbn, y_test, sensitive_dict, **de_kwargs
        )
        _report_per_attr_eo(y_test, sensitive_dict, pred_final_post)

        eo_pre   = _eo_from_binary(pred_s3,        y_test, sens_arrays_for_eo)
        eo_post  = _eo_from_binary(pred_final_post, y_test, sens_arrays_for_eo)
        acc_pre  = accuracy_score(y_test, pred_s3)
        acc_post = accuracy_score(y_test, pred_final_post)
        acc_floor = (original_baseline_acc - params["max_acc_drop"]
                     if original_baseline_acc else acc_pre - params["max_acc_drop"])

        if eo_post < eo_pre and acc_post >= acc_floor:
            pred_final = pred_final_post
            tqdm.write(f"    ✓ BBN Posterior helped: EO {eo_pre:.4f}→{eo_post:.4f} "
                       f"acc {acc_pre:.4f}→{acc_post:.4f}")
        else:
            tqdm.write(f"    ✗ BBN Posterior no help: EO {eo_pre:.4f}→{eo_post:.4f} "
                       f"— keeping Stage3 result")

        tqdm.write(f"\n  ═══ SANDWICH SUMMARY [{dataset_type.upper()}] ═══")
        tqdm.write(f"    [1] BBN Prior     : naive EO={diag_prior_raw:.4f} → soft EO={diag_prior_soft:.4f}")
        tqdm.write(f"    [2] Adversarial   : EO={diag_adv:.4f}")
        tqdm.write(f"    [3] DE Threshold  : EO={diag_s3:.4f}")
        tqdm.write(f"    [4] BBN Posterior : EO={diag_bbn_post:.4f} → after DE={eo_post:.4f}")

    acc_final      = accuracy_score(y_test, pred_final)
    fairness_final = compute_fairness_metrics(y_test, pred_final, sensitive_dict)

    del model, opt1, opt2, loader
    del X_tr_t, X_te_t, X_tr_orig_t, y_tr_t, s_tr_t
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"pred": pred_final, "acc": acc_final, **fairness_final}


# ─────────────────────────────────────────────────────────────────────────────
#  Reporting  — EO now shown as floor2 (2 d.p.)
# ─────────────────────────────────────────────────────────────────────────────

def print_results(dataset_name, baseline_results, adapter_results):
    print(f"\n{'='*70}\n{dataset_name.upper()} RESULTS\n{'='*70}")
    acc_b = baseline_results["acc"]
    acc_f = adapter_results["acc"]
    drop  = acc_b - acc_f
    tier  = ("EXCELLENT" if drop <= 0.005 else
             "GOOD"       if drop <= 0.015 else
             "ACCEPTABLE" if drop <= 0.030 else
             "HIGH"       if drop <= 0.050 else "TOO HIGH")
    print(f"Accuracy: baseline={acc_b:.4f} fair={acc_f:.4f} drop={drop:.4f} ({100*drop:.2f}%) [{tier}]")
    print("EO Metrics (floor2 — rounded DOWN to 2 d.p.):")
    for key in sorted(k for k in adapter_results if "_eo" in k):
        attr  = key.replace("_eo", "")
        # ── Change 1: use floor2 for display ─────────────────────────────────
        b_eo  = floor2(baseline_results.get(key, 0.0))
        f_eo  = floor2(adapter_results[key])
        status = "TARGET ✓" if f_eo <= TARGET_EO else ("CLOSE" if f_eo <= 0.020 else "MISS ✗")
        print(f"  {attr:14s}: {b_eo:.2f} → {f_eo:.2f} [{status}]")


# ─────────────────────────────────────────────────────────────────────────────
#  Entry point
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 70)
    print("Fairness Pipeline v4.5 — v4.4 base + floor2 EO + impossibility diagnostic")
    print(f"Changes vs v4.4: EO reported at 2 d.p. (floor2); new research-backed")
    print(f"  impossibility diagnostic (KMR16, HPS16, BHN19, ABD18, ZSD18).")
    print(f"Target: EO ≤ {TARGET_EO} | Max acc drop ≤ 4% | Device: {DEVICE}")
    print("=" * 70)

    datasets = [
        ("COMPAS",    preprocess_compas_for_fair_bbn,            "compas"),
        ("GERMAN",    preprocess_german_for_fair_bbn,            "german"),
        ("BANK",      preprocess_bank_for_fair_bbn,              "bank"),
        ("LAWSCHOOL", preprocess_lawschool_for_fair_bbn,         "lawschool"),
        ("HOSPITAL",  preprocess_diabetes_hospital_for_fair_bbn, "hospital"),
    ]

    all_results, baseline_results, all_data = {}, {}, {}

    for name, data_func, ds_key in datasets:
        print(f"\n{'='*70}\n{ds_key.upper()}\n{'='*70}")
        data = data_func()
        all_data[ds_key] = data  # store for impossibility diagnostic

        print("Training baseline MLP...")
        mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                             random_state=SEED, early_stopping=True)
        mlp.fit(data["X_train_nn"], data["y_train"])
        base_pred = mlp.predict(data["X_test_nn"])
        sens_dict = {k.replace("sens_", "").replace("_test", ""): data[k]
                     for k in data if "sens_" in k and "_test" in k}
        baseline_results[ds_key] = {
            "pred": base_pred,
            "acc":  accuracy_score(data["y_test"], base_pred),
            **compute_fairness_metrics(data["y_test"], base_pred, sens_dict),
        }
        mlp_acc = baseline_results[ds_key]["acc"]
        print(f"  MLP baseline acc: {mlp_acc:.4f}")
        del mlp
        gc.collect()

        print("\nTraining adversarial adapter + BBN sandwich v4.5...")
        adapter_results = train_model(data, ds_key, DATASET_STRATEGIES[ds_key],
                                       original_baseline_acc=mlp_acc)
        all_results[ds_key] = adapter_results
        print_results(ds_key, baseline_results[ds_key], adapter_results)
        gc.collect()

    # ── FINAL SUMMARY — EO at floor2 ─────────────────────────────────────────
    print(f"\n{'='*70}\nFINAL SUMMARY  (EO rounded DOWN to 2 decimal places)\n{'='*70}")
    for ds in ["compas", "german", "bank", "lawschool", "hospital"]:
        if ds not in all_results:
            continue
        res    = all_results[ds]
        base   = baseline_results[ds]
        # ── Change 1: floor2 for final summary ───────────────────────────────
        max_eo = floor2(max((abs(v) for k, v in res.items() if "_eo" in k), default=0))
        drop   = base["acc"] - res["acc"]
        status = ("SUCCESS ✓" if max_eo <= TARGET_EO and drop <= 0.04 else
                  "CLOSE"     if max_eo <= 0.020 and drop <= 0.04 else "MISS ✗")
        print(f"{ds.upper():12s}: acc={res['acc']:.4f} (drop={drop:+.4f}) EO={max_eo:.2f} [{status}]")

    # ── Change 2: Research-backed impossibility diagnostic ───────────────────
    fairness_impossibility_diagnostic(all_results, baseline_results, all_data)


if __name__ == "__main__":
    main()

Fairness Pipeline v4.5 — v4.4 base + floor2 EO + impossibility diagnostic
Changes vs v4.4: EO reported at 2 d.p. (floor2); new research-backed
  impossibility diagnostic (KMR16, HPS16, BHN19, ABD18, ZSD18).
Target: EO ≤ 0.01 | Max acc drop ≤ 4% | Device: cuda

COMPAS
Cached COMPAS data to /kaggle/working/cache/preproc_compas_v4.pkl
Training baseline MLP...
  MLP baseline acc: 0.6947

Training adversarial adapter + BBN sandwich v4.5...
  Features selected: 196 (proxy_thr=0.07)
  ═══ [SANDWICH-TOP] BBN PRIOR (3 rounds) ═══
    [sandwich diag] naive MLP (before prior): EO=0.2063 [ceil=0.207]


  BBN Prior rounds:   0%|          | 0/3 [00:00<?, ?it/s]

    BBN soft targets: 4937 rows → 34 unique evidence combos


  BBN unique combos:   0%|          | 0/34 [00:00<?, ?it/s]

    Round 1: mean=0.456 std=0.234
    BBN soft targets: 4937 rows → 30 unique evidence combos


...

    Round 2: mean=0.442 std=0.169
    [sandwich diag] BBN soft targets (after prior): EO=0.0376 [ceil=0.038]
    ↳ BBN Prior EO delta: 0.0377 → 0.0376 (Δ=-0.0001)
  ═══ Stage 1: Task + balance + alignment + BBN prior ═══


  Stage1:   0%|          | 0/130 [00:00<?, ?it/s]

    Stage1 early stop at epoch 110
  Stage 1 done: acc=0.6316 EO=0.0351 [ceil=0.036]
  ═══ Stage 2 [PATH A]: λ 0.15→0.8 mmd=1.5 eo_w=0 ═══


  Stage2A:   0%|          | 0/160 [00:00<?, ?it/s]

    → Lambda boost #1: max=0.920
    → Lambda boost #2: max=1.058
    → Lambda boost #3: max=1.217
    → Lambda boost #4: max=1.399
    → Lambda boost #5: max=1.609
    → Lambda boost #6: max=1.850
    → Lambda boost #7: max=2.128
  Adversary acc: 0.560 [equilibrium ✓]
    [sandwich diag] after Stage2 (adversarial): EO=0.0324 [ceil=0.033]
    [hospital_test_raw] mean=0.493 std=0.257 range=[0.001,0.999]
    [hospital_test_calibrated] mean=0.442 std=0.108 range=[0.248,0.656]
    [hospital_train_calibrated] mean=0.441 std=0.112 range=[0.248,0.657]
  Stage 2 done: EO=0.0321 [ceil=0.033] acc=0.6100
  ═══ Stage 3: DE Group-threshold calibration ═══
    DE threshold: floor=0.5928 baseline_EO=0.0338 opt_t=0.530
    [seed=42] DE EO=0.0022 [ceil=0.003] (evals=7565)
    ✓ Target met at seed=42 — skipping remaining seeds.
    DE result: EO=0.0022 [ceil=0.003] acc=0.5948
    race: TPR=(0.186,0.188) FPR=(0.083,0.080) EO=0.0022 [ceil=0.003]
    sex: TPR=(0.187,0.187) FPR=(0.083,0.081) EO=0.0020 [ceil